# HungerHub Analytics - End-to-End Oracle Analysis

##  Project Overview
This notebook demonstrates a complete **Oracle → Python → Graphics** workflow for HungerHub food distribution analytics.

### Complete Workflow:
1. ** Oracle Database Connection**: Direct connection to Choice & AgencyExpress databases
2. ** Data Extraction**: SQL queries to extract raw data from Oracle tables
3. ** ETL Pipeline**: Transform and standardize data, store locally in CSV/Parquet
4. ** Exploratory Data Analysis**: Uni-, bi-, and multi-variable statistical analysis
5. ** Visual Reports**: Interactive Plotly Express charts and dashboards
6. ** Data Quality**: Identify issues and execute cleaning strategies
7. ** Business Insights**: Draw actionable conclusions for hunger relief operations

### Oracle Data Sources:
- **Choice Database**: Service delivery, client demographics, food distribution
- **AgencyExpress Database**: Donor management, agency operations, geographical data

---

## 1. Environment Setup & Library Imports

Loading all required libraries for the complete Oracle-to-Graphics pipeline.

** Prerequisites:**
- Jupyter kernel: `HungerHub POC` (with Oracle libraries installed)
- Virtual environment activated with cx_Oracle, oracledb, scipy, tqdm

In [46]:
# Core data manipulation and analysis
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Oracle database connectivity
import cx_Oracle
import oracledb
import pyodbc
from sqlalchemy import create_engine
import sqlalchemy

# File I/O and configuration
import os
import sys
import json
from pathlib import Path
from dotenv import load_dotenv

# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "plotly_white"

# Statistical analysis and visualization
from scipy import stats as scipy_stats
import seaborn as sns
import matplotlib.pyplot as plt

# Progress tracking
from tqdm import tqdm

print("✅ All libraries loaded successfully!")
print(f" Oracle-to-Graphics analysis started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f" Oracle cx_Oracle version: {cx_Oracle.version}")
print(f" Oracle oracledb version: {oracledb.__version__}")

✅ All libraries loaded successfully!
 Oracle-to-Graphics analysis started: 2025-08-09 16:56:17
 Oracle cx_Oracle version: 8.3.0
 Oracle oracledb version: 3.3.0


In [47]:
# Configure display and analysis options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)
np.random.seed(42)  # For reproducible results

# Set up project paths
project_root = Path('../')
data_path = project_root / 'data'
config_path = project_root / 'config'
src_path = project_root / 'src'
output_path = Path('notebook_output')
output_path.mkdir(exist_ok=True)

# Add src to Python path for custom modules
sys.path.append(str(src_path))

print(f" Project root: {project_root.absolute()}")
print(f" Data path: {data_path.absolute()}")
print(f" Config path: {config_path.absolute()}")
print(f" Output path: {output_path.absolute()}")

 Project root: /home/tagazureuser/cgorr/2week_poc_execution/hungerhub_poc/notebooks/..
 Data path: /home/tagazureuser/cgorr/2week_poc_execution/hungerhub_poc/notebooks/../data
 Config path: /home/tagazureuser/cgorr/2week_poc_execution/hungerhub_poc/notebooks/../config
 Output path: /home/tagazureuser/cgorr/2week_poc_execution/hungerhub_poc/notebooks/notebook_output


In [48]:
# Analysis execution timer - START
analysis_start_time = datetime.now()
print(f"Analysis execution started at: {analysis_start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Notebook type: SAMPLE DATA ANALYSIS")
print("-" * 50)

Analysis execution started at: 2025-08-09 16:56:17
Notebook type: SAMPLE DATA ANALYSIS
--------------------------------------------------


In [49]:
# Create notebook output directory structure
notebook_outputs = {
    'base': Path('notebook_output'),
    'csv': Path('notebook_output/csv'),
    'parquet': Path('notebook_output/parquet'),
    'reports': Path('notebook_output/reports'),
    'visualizations': Path('notebook_output/visualizations')
}

# Create all output directories
for name, path in notebook_outputs.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f" Created: {path}")

# Update output_path to use notebook directory
output_path = notebook_outputs['base']
print(f"\n✅ All notebook outputs will be saved to: {output_path.absolute()}")
print(f" Available subdirectories: {list(notebook_outputs.keys())}")

 Created: notebook_output
 Created: notebook_output/csv
 Created: notebook_output/parquet
 Created: notebook_output/reports
 Created: notebook_output/visualizations

✅ All notebook outputs will be saved to: /home/tagazureuser/cgorr/2week_poc_execution/hungerhub_poc/notebooks/notebook_output
 Available subdirectories: ['base', 'csv', 'parquet', 'reports', 'visualizations']


In [50]:
# Load environment variables and Oracle connection settings
env_file = config_path / '.env'
if env_file.exists():
    load_dotenv(env_file)
    print("✅ Environment variables loaded from .env file")
    
    # Display available Oracle connection parameters (without showing sensitive data)
    oracle_params = ['ORACLE_HOST', 'ORACLE_PORT', 'ORACLE_SERVICE', 'ORACLE_USER']
    available_params = []
    for param in oracle_params:
        if os.getenv(param):
            available_params.append(param)
    
    if available_params:
        print(f" Oracle connection parameters available: {', '.join(available_params)}")
        # Test Oracle Instant Client availability
        try:
            cx_Oracle.clientversion()
            print("✅ Oracle Instant Client is available")
        except cx_Oracle.DatabaseError as e:
            print(f" Oracle Instant Client issue: {e}")
    else:
        print(" No Oracle connection parameters found in environment")
else:
    print(" .env file not found - will use existing processed data for analysis")

print("\n Environment setup complete! Ready for Oracle database connection and analysis.")

✅ Environment variables loaded from .env file
 No Oracle connection parameters found in environment

 Environment setup complete! Ready for Oracle database connection and analysis.


## 2. Oracle Database Connection & Data Extraction

Connecting to the actual HungerHub Oracle databases using the existing project configuration.

### Database Architecture:
- **Choice Sandbox**: Donations and Choice program data (`52.43.135.66:1521/staging`)
- **AgencyExpress Sandbox**: Agency operations data (`52.43.135.66:1521/staging`)

### Connection Strategy:
- Use the same connection parameters as existing project modules
- Test connectivity to both databases
- Extract sample data from priority tables
- Fall back to existing processed data if connections fail

In [51]:
# Oracle connection setup based on existing project configuration
class HungerHubOracleConnector:
    """Oracle database connector matching existing project setup"""
    
    def __init__(self):
        # Database configurations from .env file
        self.databases = {
            'choice': {
                'host': os.getenv('CHOICE_ORACLE_HOST'),
                'port': os.getenv('CHOICE_ORACLE_PORT', '1521'),
                'service': os.getenv('CHOICE_ORACLE_SERVICE_NAME'),
                'user': os.getenv('CHOICE_USERNAME'),
                'password': os.getenv('CHOICE_PASSWORD'),
                'name': 'Choice Sandbox'
            },
            'agency': {
                'host': os.getenv('AGENCY_ORACLE_HOST'),
                'port': os.getenv('AGENCY_ORACLE_PORT', '1521'),
                'service': os.getenv('AGENCY_ORACLE_SERVICE_NAME'),
                'user': os.getenv('AGENCY_USERNAME'),
                'password': os.getenv('AGENCY_PASSWORD'),
                'name': 'Agency Sandbox'
            }
        }
    
    def test_connection(self, db_key):
        """Test connection to a specific database"""
        config = self.databases[db_key]
        
        print(f"\nTesting {config['name']} Database:")
        print("-" * 40)
        
        # Check configuration
        missing_config = [k for k, v in config.items() if not v]
        if missing_config:
            print(f"❌ Missing configuration: {', '.join(missing_config)}")
            return None
        
        try:
            # Create DSN connection string
            dsn = cx_Oracle.makedsn(
                config['host'], 
                config['port'], 
                service_name=config['service']
            )
            
            print(f"Connecting to: {config['host']}:{config['port']}/{config['service']}")
            print(f"User: {config['user']}")
            
            # Establish connection
            connection = cx_Oracle.connect(
                config['user'], 
                config['password'], 
                dsn
            )
            
            print("✅ Connected successfully!")
            print(f"Oracle Version: {connection.version}")
            
            # Test basic query
            cursor = connection.cursor()
            cursor.execute("SELECT USER, SYSDATE FROM DUAL")
            user, sysdate = cursor.fetchone()
            
            print(f"Connected as: {user}")
            print(f"Server time: {sysdate}")
            
            # Test table access
            try:
                cursor.execute("SELECT COUNT(*) FROM user_tables")
                table_count = cursor.fetchone()[0]
                print(f"Accessible tables: {table_count}")
            except Exception as e:
                print(f"⚠️ Table query warning: {e}")
            
            cursor.close()
            return connection
            
        except cx_Oracle.Error as error:
            print(f"❌ Oracle connection failed: {error}")
            return None
        except Exception as error:
            print(f"❌ Unexpected error: {error}")
            return None
    def get_table_list(self, connection):
        """Get list of accessible tables"""
        try:
            cursor = connection.cursor()
            cursor.execute("""
                SELECT table_name, num_rows 
                FROM user_tables 
                ORDER BY table_name
            """)
            results = cursor.fetchall()
            cursor.close()
            
            df = pd.DataFrame(results, columns=['TABLE_NAME', 'NUM_ROWS'])
            return df
        except Exception as e:
            print(f"❌ Error getting table list: {e}")
            return None

# Initialize Oracle connector
oracle_connector = HungerHubOracleConnector()
print("✅ Oracle connector initialized with project configuration")

✅ Oracle connector initialized with project configuration


In [52]:
# Test Oracle database connections
oracle_connections = {}

print("Testing Oracle Database Connections")
print("=" * 50)

# Test Choice database
choice_conn = oracle_connector.test_connection('choice')
if choice_conn:
    oracle_connections['choice'] = choice_conn
    print("  ✅ Choice database connection successful")
else:
    print("  ❌ Choice database connection failed")

# Test Agency database
agency_conn = oracle_connector.test_connection('agency')
if agency_conn:
    oracle_connections['agency'] = agency_conn
    print("  ✅ Agency database connection successful")
else:
    print("  ❌ Agency database connection failed")

print(f"\nConnection Summary: {len(oracle_connections)}/2 databases connected")

if oracle_connections:
    print("✅ At least one Oracle database connection successful!")
    print("Ready to proceed with data extraction.")
    
    # Simple 5-second delay to let connections stabilize
    import time
    print("\nLetting connections stabilize...")
    time.sleep(5)
    print("✅ Connections ready for use")
else:
    print("⚠️ No Oracle connections available - will use existing processed data")

Testing Oracle Database Connections

Testing Choice Sandbox Database:
----------------------------------------
Connecting to: 52.43.135.66:1521/staging
User: rwtxn_46
✅ Connected successfully!
Oracle Version: 11.2.0.1.0
Connected as: RWTXN_46
Server time: 2025-08-09 11:57:03
Accessible tables: 283
  ✅ Choice database connection successful

Testing Agency Sandbox Database:
----------------------------------------
Connecting to: 52.43.135.66:1521/staging
User: tran_user
✅ Connected successfully!
Oracle Version: 11.2.0.1.0
Connected as: TRAN_USER
Server time: 2025-08-09 11:57:04
Accessible tables: 367
  ✅ Agency database connection successful

Connection Summary: 2/2 databases connected
✅ At least one Oracle database connection successful!
Ready to proceed with data extraction.

Letting connections stabilize...
✅ Connections ready for use


In [53]:
# Extract sample data from priority tables
def extract_table_sample(connection, table_name, sample_size=1000):
    """Extract sample data from Oracle table"""
    try:
        # Oracle-specific query with ROWNUM for performance
        query = f"""
        SELECT * FROM {table_name} 
        WHERE ROWNUM <= {sample_size}
        """
        
        cursor = connection.cursor()
        cursor.execute(query)
        results = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description]
        cursor.close()
        
        df = pd.DataFrame(results, columns=columns)
        print(f"    ✅ Extracted {len(df)} rows, {len(df.columns)} columns")
        return df
        
    except Exception as e:
        print(f"    ❌ Error extracting {table_name}: {e}")
        return None

# Priority tables based on existing project structure
priority_tables = {
    'choice': [
        'DOCUMENTHEADER',
        'AMX_DONATION_HEADER', 
        'AMX_DONATION_LINES',
        'RW_ORG'
    ],
    'agency': [
        'DOCUMENTHEADER',
        'DOCUMENTLINES',
        'RW_ORG'
    ]
}

extracted_oracle_data = {}

if oracle_connections:
    print(" Extracting Sample Data from Priority Tables")
    print("=" * 50)
    
    for db_key, connection in oracle_connections.items():
        print(f"\n Extracting from {oracle_connector.databases[db_key]['name']}:")
        
        # Get available tables for this database
        available_tables = set()
        # Extract all priority tables (no pre-validation needed)
        
        # Extract priority tables that exist
        for table_name in priority_tables.get(db_key, []):
            if table_name in available_tables or not available_tables:
                print(f"   Extracting {table_name}...")
                df = extract_table_sample(connection, table_name, sample_size=500)
                if df is not None:
                    key = f"{db_key}_{table_name}"
                    extracted_oracle_data[key] = df
                    
                    # Save to notebook output directory
                    output_file = output_path / f"{key}_sample.csv"
                    df.to_csv(output_file, index=False)
                    print(f"     Saved to {output_file.name}")
            else:
                print(f"   Table {table_name} not found in {db_key} database")
    
    print(f"\n✅ Successfully extracted {len(extracted_oracle_data)} datasets from Oracle")
else:
    print(" Loading existing processed data as fallback...")
    
    # Load existing processed data
    processed_data_path = data_path / 'processed'
    for file_path in processed_data_path.rglob('*.csv'):
        if 'unified_real' in str(file_path) or '_sample' in str(file_path.name):
            table_name = file_path.stem.replace('_sample', '')
            try:
                df = pd.read_csv(file_path)
                extracted_oracle_data[table_name] = df
                print(f"   Loaded {table_name}: {len(df)} rows × {len(df.columns)} columns")
            except Exception as e:
                print(f"  ❌ Error loading {file_path.name}: {e}")
    
    print(f"\n✅ Loaded {len(extracted_oracle_data)} datasets from existing files")

# Display summary of extracted data
if extracted_oracle_data:
    print(f"\n Data Extraction Summary:")
    print("-" * 60)
    total_rows = 0
    for dataset_name, df in extracted_oracle_data.items():
        rows, cols = df.shape
        total_rows += rows
        print(f"   {dataset_name:<30} {rows:>6} rows × {cols:>3} columns")
    
    print(f"\n Total extracted records: {total_rows:,} rows across {len(extracted_oracle_data)} tables")
else:
    print("❌ No data available for analysis")

 Extracting Sample Data from Priority Tables

 Extracting from Choice Sandbox:
   Extracting DOCUMENTHEADER...
    ✅ Extracted 500 rows, 51 columns
     Saved to choice_DOCUMENTHEADER_sample.csv
   Extracting AMX_DONATION_HEADER...
    ✅ Extracted 500 rows, 30 columns
     Saved to choice_AMX_DONATION_HEADER_sample.csv
   Extracting AMX_DONATION_LINES...
    ✅ Extracted 500 rows, 26 columns
     Saved to choice_AMX_DONATION_LINES_sample.csv
   Extracting RW_ORG...
    ✅ Extracted 500 rows, 44 columns
     Saved to choice_RW_ORG_sample.csv

 Extracting from Agency Sandbox:
   Extracting DOCUMENTHEADER...
    ✅ Extracted 500 rows, 35 columns
     Saved to agency_DOCUMENTHEADER_sample.csv
   Extracting DOCUMENTLINES...
    ✅ Extracted 500 rows, 57 columns
     Saved to agency_DOCUMENTLINES_sample.csv
   Extracting RW_ORG...
    ✅ Extracted 500 rows, 50 columns
     Saved to agency_RW_ORG_sample.csv

✅ Successfully extracted 7 datasets from Oracle

 Data Extraction Summary:
---------------

## 2.5 Database Schema Documentation

Comprehensive documentation of the HungerHub Oracle database schemas, table structures, and data characteristics.

### Database Architecture Overview:
- **Choice Sandbox**: Primary food distribution and client management system
- **AgencyExpress Sandbox**: Donor management and agency operations system

This section provides detailed schema analysis, record counts, data types, and relationship mapping between the two database systems.

In [54]:
# Comprehensive Database Documentation Generator
class DatabaseDocumentor:
    """Generate comprehensive database schema documentation"""
    
    def __init__(self, oracle_connections):
        self.connections = oracle_connections
        self.schema_info = {}
        self.documentation = {}
    
    def get_detailed_schema_info(self, connection, db_name):
        """Get comprehensive schema information"""
        try:
            cursor = connection.cursor()
            
            # Get table information with detailed stats
            table_query = """
            SELECT 
                table_name,
                num_rows,
                blocks,
                avg_row_len,
                last_analyzed,
                partitioned
            FROM user_tables 
            ORDER BY NVL(num_rows, 0) DESC
            """
            
            cursor.execute(table_query)
            tables_data = cursor.fetchall()
            
            tables_info = []
            for row in tables_data:
                table_info = {
                    'table_name': row[0],
                    'num_rows': row[1] if row[1] is not None else 0,
                    'blocks': row[2] if row[2] is not None else 0,
                    'avg_row_len': row[3] if row[3] is not None else 0,
                    'last_analyzed': row[4],
                    'partitioned': row[5]
                }
                tables_info.append(table_info)
            
            # Get column information for key tables (top 10 by size)
            top_tables = sorted(tables_info, key=lambda x: x['num_rows'], reverse=True)[:10]
            
            detailed_tables = {}
            for table in top_tables:
                table_name = table['table_name']
                
                # Get column details
                column_query = """
                SELECT 
                    column_name,
                    data_type,
                    data_length,
                    data_precision,
                    data_scale,
                    nullable,
                    column_id
                FROM user_tab_columns 
                WHERE table_name = :table_name
                ORDER BY column_id
                """
                
                cursor.execute(column_query, {'table_name': table_name})
                columns_data = cursor.fetchall()
                
                columns_info = []
                for col_row in columns_data:
                    col_info = {
                        'column_name': col_row[0],
                        'data_type': col_row[1],
                        'data_length': col_row[2],
                        'data_precision': col_row[3],
                        'data_scale': col_row[4],
                        'nullable': col_row[5],
                        'column_id': col_row[6]
                    }
                    columns_info.append(col_info)
                
                table['columns'] = columns_info
                detailed_tables[table_name] = table
            
            cursor.close()
            
            return {
                'database_name': db_name,
                'total_tables': len(tables_info),
                'total_records': sum(t['num_rows'] for t in tables_info),
                'tables_summary': tables_info,
                'detailed_tables': detailed_tables
            }
            
        except Exception as e:
            print(f"Error getting schema info for {db_name}: {e}")
            return None
    
    def generate_database_summary(self, schema_info):
        """Generate human-readable database summary"""
        db_name = schema_info['database_name']
        
        print(f"\n{'='*60}")
        print(f"DATABASE: {db_name.upper()}")
        print(f"{'='*60}")
        
        print(f"\nDATABASE OVERVIEW:")
        print(f"  Total Tables: {schema_info['total_tables']:,}")
        print(f"  Total Records: {schema_info['total_records']:,}")
        
        # Table size distribution
        tables = schema_info['tables_summary']
        large_tables = [t for t in tables if t['num_rows'] > 10000]
        medium_tables = [t for t in tables if 1000 <= t['num_rows'] <= 10000]
        small_tables = [t for t in tables if 0 < t['num_rows'] < 1000]
        empty_tables = [t for t in tables if t['num_rows'] == 0]
        
        print(f"\nTABLE SIZE DISTRIBUTION:")
        print(f"  Large (>10K records): {len(large_tables)} tables")
        print(f"  Medium (1K-10K records): {len(medium_tables)} tables")
        print(f"  Small (<1K records): {len(small_tables)} tables")
        print(f"  Empty (0 records): {len(empty_tables)} tables")
        
        # Top tables by size
        print(f"\nTOP 10 TABLES BY RECORD COUNT:")
        top_tables = sorted(tables, key=lambda x: x['num_rows'], reverse=True)[:10]
        for i, table in enumerate(top_tables, 1):
            avg_len = f"({table['avg_row_len']} bytes avg)" if table['avg_row_len'] else ""
            print(f"  {i:2d}. {table['table_name']:<30} {table['num_rows']:>10,} rows {avg_len}")
        
        return {
            'large_tables': len(large_tables),
            'medium_tables': len(medium_tables),
            'small_tables': len(small_tables),
            'empty_tables': len(empty_tables)
        }
    
    def generate_table_details(self, schema_info, max_tables=5):
        """Generate detailed table documentation"""
        db_name = schema_info['database_name']
        detailed_tables = schema_info['detailed_tables']
        
        print(f"\nDETAILED TABLE SCHEMAS ({db_name.upper()}):")
        print(f"{'='*50}")
        
        # Show details for top tables
        table_names = sorted(detailed_tables.keys(), 
                           key=lambda x: detailed_tables[x]['num_rows'], 
                           reverse=True)[:max_tables]
        
        for table_name in table_names:
            table = detailed_tables[table_name]
            
            print(f"\nTABLE: {table_name}")
            print("-" * 40)
            print(f"Records: {table['num_rows']:,}")
            print(f"Columns: {len(table['columns'])}")
            print(f"Avg Row Length: {table['avg_row_len']} bytes")
            print(f"Last Analyzed: {table['last_analyzed']}")
            
            print(f"\nCOLUMN STRUCTURE:")
            print(f"{'#':<3} {'Column Name':<25} {'Data Type':<15} {'Length':<8} {'Nullable':<8}")
            print("-" * 65)
            
            for col in table['columns'][:15]:  # Show first 15 columns
                col_type = col['data_type']
                if col['data_precision'] and col['data_scale']:
                    col_type += f"({col['data_precision']},{col['data_scale']})"
                elif col['data_length'] and col['data_type'] in ['VARCHAR2', 'CHAR']:
                    col_type += f"({col['data_length']})"
                
                nullable = "YES" if col['nullable'] == 'Y' else "NO"
                
                print(f"{col['column_id']:<3} {col['column_name']:<25} {col_type:<15} {col['data_length'] or '':<8} {nullable:<8}")
            
            if len(table['columns']) > 15:
                print(f"     ... and {len(table['columns']) - 15} more columns")
    
    def document_all_databases(self):
        """Generate complete database documentation"""
        print("COMPREHENSIVE DATABASE SCHEMA DOCUMENTATION")
        print("=" * 60)
        
        all_stats = {}
        
        for db_key, connection in self.connections.items():
            db_name = oracle_connector.databases[db_key]['name']
            
            print(f"\nAnalyzing {db_name} schema...")
            schema_info = self.get_detailed_schema_info(connection, db_name)
            
            if schema_info:
                # Generate summary
                stats = self.generate_database_summary(schema_info)
                all_stats[db_name] = stats
                
                # Generate detailed table documentation
                self.generate_table_details(schema_info)
                
                # Store for later use
                self.schema_info[db_key] = schema_info
        
        return all_stats

# Initialize database documentation if Oracle connections are available
if 'oracle_connections' in globals() and oracle_connections:
    db_documentor = DatabaseDocumentor(oracle_connections)
    print("✅ Database documentation toolkit initialized")
    print(f"Ready to document {len(oracle_connections)} Oracle databases")
else:
    print("❌ Database documentation requires active Oracle connections")
    print("Please run Section 2 Oracle connections first")

✅ Database documentation toolkit initialized
Ready to document 2 Oracle databases


In [55]:
# Generate Complete Database Documentation
if 'db_documentor' in globals():
    database_stats = db_documentor.document_all_databases()
    
    # Generate cross-database comparison
    if len(database_stats) > 1:
        print(f"\n\nCROSS-DATABASE COMPARISON")
        print("=" * 40)
        
        for db_name, stats in database_stats.items():
            schema_info = None
            for key, info in db_documentor.schema_info.items():
                if info['database_name'] == db_name:
                    schema_info = info
                    break
            
            if schema_info:
                print(f"\n{db_name}:")
                print(f"  Total Tables: {schema_info['total_tables']}")
                print(f"  Total Records: {schema_info['total_records']:,}")
                print(f"  Large Tables: {stats['large_tables']}")
                print(f"  Empty Tables: {stats['empty_tables']}")
    
    print(f"\n✅ Database documentation complete!")
else:
    print("❌ Database documentation toolkit not available")

COMPREHENSIVE DATABASE SCHEMA DOCUMENTATION

Analyzing Choice Sandbox schema...

DATABASE: CHOICE SANDBOX

DATABASE OVERVIEW:
  Total Tables: 283
  Total Records: 7,646,604

TABLE SIZE DISTRIBUTION:
  Large (>10K records): 23 tables
  Medium (1K-10K records): 23 tables
  Small (<1K records): 132 tables
  Empty (0 records): 105 tables

TOP 10 TABLES BY RECORD COUNT:
   1. RW_FLEX                         4,393,524 rows (49 bytes avg)
   2. RW_PO_FLEX                        578,995 rows (54 bytes avg)
   3. RW_CONTACT_INFO                   566,783 rows (157 bytes avg)
   4. ACSHARES_ARCHIVE                  554,550 rows (100 bytes avg)
   5. RW_SHPMNT_METHOD                  231,675 rows (35 bytes avg)
   6. RW_ORDR_ITM_SCHDL                 230,282 rows (132 bytes avg)
   7. RW_ORDER_ITEM                     230,282 rows (236 bytes avg)
   8. RW_DOC_TRANS                      108,861 rows (55 bytes avg)
   9. BIDDINGSESSION_SNAPSHOT           102,470 rows (18 bytes avg)
  10. RW_ORDER_S

### Database Documentation Summary

The above analysis provides:

1. **Schema Overview**: Total tables and records per database
2. **Table Size Distribution**: Large, medium, small, and empty tables
3. **Top Tables**: Ranked by record count with size metrics
4. **Detailed Schema**: Column structures for largest tables
5. **Cross-Database Comparison**: Relative sizes and characteristics

This documentation serves as a reference for:
- Understanding data volume and distribution
- Identifying key tables for analysis
- Planning data extraction and ETL strategies
- Assessing data quality and completeness

## 3. ETL Pipeline Implementation

Implementing Extract, Transform, and Load processes to clean and standardize the Oracle data for analysis.

### ETL Strategy:
1. **Extract**: Raw data from Oracle (completed in Section 2) or existing processed files
2. **Transform**: Clean, standardize, and unify data across both databases
3. **Load**: Store processed data in optimized formats (CSV and Parquet)

### Data Quality Focus:
- **Standardize column names** and data types
- **Handle missing values** and duplicates
- **Unify date formats** and currency values
- **Create unified datasets** for cross-database analysis

### Output Formats:
- **CSV files**: Human-readable for inspection
- **Parquet files**: Optimized for fast analysis and visualization

In [56]:
# ETL Pipeline implementation
class HungerHubETLPipeline:
    """ETL Pipeline for HungerHub Oracle data processing"""
    
    def __init__(self, output_directory):
        self.output_dir = Path(output_directory)
        self.output_dir.mkdir(exist_ok=True)
        
        # Create subdirectories for organized output
        (self.output_dir / 'csv').mkdir(exist_ok=True)
        (self.output_dir / 'parquet').mkdir(exist_ok=True)
        
        self.transformation_log = []
        
    def log_transformation(self, step, details):
        """Log transformation steps"""
        timestamp = datetime.now().strftime('%H:%M:%S')
        log_entry = f"[{timestamp}] {step}: {details}"
        self.transformation_log.append(log_entry)
        print(f"   {step}: {details}")
    
    def standardize_column_names(self, df, dataset_name):
        """Standardize column names across datasets"""
        original_columns = df.columns.tolist()
        
        # Convert to lowercase and replace spaces/special chars
        df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('[^a-z0-9_]', '', regex=True)
        
        # Common column mappings for consistency
        column_mappings = {
            'id': 'record_id',
            'name': 'entity_name',
            'date': 'record_date',
            'amount': 'transaction_amount',
            'qty': 'quantity',
            'description': 'item_description'
        }
        
        # Apply mappings if columns exist
        for old_name, new_name in column_mappings.items():
            if old_name in df.columns:
                df.rename(columns={old_name: new_name}, inplace=True)
        
        renamed_count = sum(1 for old, new in zip(original_columns, df.columns) if old != new)
        self.log_transformation(f"Column Standardization ({dataset_name})", 
                              f"Standardized {renamed_count}/{len(df.columns)} column names")
        
        return df
    
    def clean_missing_values(self, df, dataset_name):
        """Handle missing values strategically"""
        initial_shape = df.shape
        missing_summary = df.isnull().sum()
        missing_cols = missing_summary[missing_summary > 0]
        
        if len(missing_cols) > 0:
            self.log_transformation(f"Missing Values Analysis ({dataset_name})", 
                                  f"Found missing data in {len(missing_cols)} columns")
            
            for col in missing_cols.index:
                missing_pct = (missing_cols[col] / len(df)) * 100
                print(f"     {col}: {missing_cols[col]} missing ({missing_pct:.1f}%)")
                
                # Strategic missing value handling
                if missing_pct > 90:
                    # Drop columns that are mostly empty
                    df.drop(columns=[col], inplace=True)
                    print(f"     Dropped {col} (>90% missing)")
                elif df[col].dtype in ['object', 'string']:
                    # Fill string columns with 'Unknown'
                    df[col].fillna('Unknown', inplace=True)
                elif df[col].dtype in ['int64', 'float64']:
                    # Fill numeric columns with median
                    df[col].fillna(df[col].median(), inplace=True)
                elif df[col].dtype == 'datetime64[ns]':
                    # Fill date columns with a default date
                    df[col].fillna(pd.Timestamp('1900-01-01'), inplace=True)
        
        final_shape = df.shape
        self.log_transformation(f"Missing Values Cleanup ({dataset_name})", 
                              f"Shape changed from {initial_shape} to {final_shape}")
        
        return df
    
    def standardize_data_types(self, df, dataset_name):
        """Standardize data types for consistency"""
        type_changes = 0
        
        for col in df.columns:
            original_type = str(df[col].dtype)
            
            # Convert date-like columns
            if 'date' in col.lower() or 'time' in col.lower():
                try:
                    df[col] = pd.to_datetime(df[col], errors='coerce')
                    if str(df[col].dtype) != original_type:
                        type_changes += 1
                except:
                    pass
            
            # Convert amount/price columns to float
            elif any(keyword in col.lower() for keyword in ['amount', 'price', 'cost', 'value']):
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                    if str(df[col].dtype) != original_type:
                        type_changes += 1
                except:
                    pass
            
            # Convert ID columns to string for consistency
            elif 'id' in col.lower() and df[col].dtype != 'object':
                try:
                    df[col] = df[col].astype(str)
                    if 'object' != original_type:
                        type_changes += 1
                except:
                    pass
        
        self.log_transformation(f"Data Type Standardization ({dataset_name})", 
                              f"Converted {type_changes} column types")
        
        return df
    
    def remove_duplicates(self, df, dataset_name):
        """Remove duplicate records"""
        initial_count = len(df)
        df_cleaned = df.drop_duplicates()
        duplicates_removed = initial_count - len(df_cleaned)
        
        if duplicates_removed > 0:
            self.log_transformation(f"Duplicate Removal ({dataset_name})", 
                                  f"Removed {duplicates_removed} duplicate records")
        else:
            self.log_transformation(f"Duplicate Check ({dataset_name})", 
                                  f"No duplicates found")
        
        return df_cleaned
    
    def process_dataset(self, df, dataset_name):
        """Apply full ETL transformation to a dataset"""
        print(f"\n Processing {dataset_name}...")
        print(f"   Input: {df.shape[0]} rows × {df.shape[1]} columns")
        
        # Apply transformations in sequence
        df_processed = df.copy()
        df_processed = self.standardize_column_names(df_processed, dataset_name)
        df_processed = self.clean_missing_values(df_processed, dataset_name)
        df_processed = self.standardize_data_types(df_processed, dataset_name)
        df_processed = self.remove_duplicates(df_processed, dataset_name)
        
        print(f"  ✅ Output: {df_processed.shape[0]} rows × {df_processed.shape[1]} columns")
        
        return df_processed
    
    def save_dataset(self, df, dataset_name, formats=['csv', 'parquet']):
        """Save processed dataset in specified formats"""
        files_saved = []
        
        for format_type in formats:
            if format_type == 'csv':
                file_path = self.output_dir / 'csv' / f"{dataset_name}_processed.csv"
                df.to_csv(file_path, index=False)
                files_saved.append(file_path)
            
            elif format_type == 'parquet':
                file_path = self.output_dir / 'parquet' / f"{dataset_name}_processed.parquet"
                df.to_parquet(file_path, index=False)
                files_saved.append(file_path)
        
        self.log_transformation(f"Data Export ({dataset_name})", 
                              f"Saved in {len(files_saved)} formats")
        
        return files_saved
    
    def generate_processing_report(self):
        """Generate ETL processing report"""
        report = {
            'timestamp': datetime.now().isoformat(),
            'transformations': self.transformation_log,
            'total_steps': len(self.transformation_log)
        }
        
        report_file = self.output_dir / 'etl_processing_report.json'
        with open(report_file, 'w') as f:
            json.dump(report, f, indent=2)
        
        print(f"\n ETL Processing Report saved to: {report_file}")
        return report

# Initialize ETL Pipeline
etl_pipeline = HungerHubETLPipeline(output_path)
print("✅ ETL Pipeline initialized")
print(f" Output directory: {output_path.absolute()}")

✅ ETL Pipeline initialized
 Output directory: /home/tagazureuser/cgorr/2week_poc_execution/hungerhub_poc/notebooks/notebook_output


In [57]:
# Process extracted Oracle data through ETL pipeline
processed_datasets = {}

if extracted_oracle_data:
    print(" Starting ETL Processing of Oracle Data")
    print("=" * 50)
    
    # Process each extracted dataset
    for dataset_name, raw_df in tqdm(extracted_oracle_data.items(), desc="Processing datasets"):
        if len(raw_df) > 0:  # Only process non-empty datasets
            try:
                # Apply ETL transformations
                processed_df = etl_pipeline.process_dataset(raw_df, dataset_name)
                
                # Save processed data
                saved_files = etl_pipeline.save_dataset(processed_df, dataset_name)
                
                # Store in memory for analysis
                processed_datasets[dataset_name] = processed_df
                
                # Display sample of processed data
                print(f"   Sample of processed {dataset_name}:")
                print(f"     Columns: {list(processed_df.columns[:5])}{'...' if len(processed_df.columns) > 5 else ''}")
                if len(processed_df) > 0:
                    print(f"     Data types: {dict(processed_df.dtypes.head())}")
                
            except Exception as e:
                print(f"  ❌ Error processing {dataset_name}: {e}")
        else:
            print(f"   Skipping empty dataset: {dataset_name}")
    
    print(f"\n✅ ETL Processing Complete: {len(processed_datasets)} datasets processed")
else:
    print(" No extracted data available for ETL processing")

 Starting ETL Processing of Oracle Data


Processing datasets:  29%|██▊       | 2/7 [00:00<00:00, 19.53it/s]


 Processing choice_DOCUMENTHEADER...
   Input: 500 rows × 51 columns
   Column Standardization (choice_DOCUMENTHEADER): Standardized 51/51 column names
   Missing Values Analysis (choice_DOCUMENTHEADER): Found missing data in 46 columns
     sourceorg: 4 missing (0.8%)
     userid: 440 missing (88.0%)
     createddate: 447 missing (89.4%)
     expirydate: 1 missing (0.2%)
     modifieddate: 500 missing (100.0%)
     Dropped modifieddate (>90% missing)
     warehouseid: 452 missing (90.4%)
     Dropped warehouseid (>90% missing)
     prop1: 195 missing (39.0%)
     prop2: 38 missing (7.6%)
     prop3: 29 missing (5.8%)
     prop4: 54 missing (10.8%)
     prop5: 9 missing (1.8%)
     prop6: 181 missing (36.2%)
     prop7: 6 missing (1.2%)
     prop8: 5 missing (1.0%)
     prop9: 5 missing (1.0%)
     prop10: 5 missing (1.0%)
     prop11: 5 missing (1.0%)
     prop12: 5 missing (1.0%)
     prop13: 5 missing (1.0%)
     prop14: 500 missing (100.0%)
     Dropped prop14 (>90% missing)
     

Processing datasets: 100%|██████████| 7/7 [00:00<00:00, 24.73it/s]

     Dropped prop12 (>90% missing)
     prop13: 500 missing (100.0%)
     Dropped prop13 (>90% missing)
     prop14: 500 missing (100.0%)
     Dropped prop14 (>90% missing)
     prop16: 500 missing (100.0%)
     Dropped prop16 (>90% missing)
     prop17: 500 missing (100.0%)
     Dropped prop17 (>90% missing)
     prop18: 500 missing (100.0%)
     Dropped prop18 (>90% missing)
     prop19: 500 missing (100.0%)
     Dropped prop19 (>90% missing)
     prop20: 500 missing (100.0%)
     Dropped prop20 (>90% missing)
     notify: 481 missing (96.2%)
     Dropped notify (>90% missing)
     prop15: 500 missing (100.0%)
     Dropped prop15 (>90% missing)
     acceptedquantity: 6 missing (1.2%)
     accepteddate: 6 missing (1.2%)
     receiptedquantity: 500 missing (100.0%)
     Dropped receiptedquantity (>90% missing)
     receipteddate: 500 missing (100.0%)
     Dropped receipteddate (>90% missing)
     destinationorgname: 500 missing (100.0%)
     Dropped destinationorgname (>90% missing)
  

In [58]:
# Create unified datasets for cross-database analysis
def create_unified_dataset(datasets, dataset_type, etl_pipeline):
    """Create unified dataset by combining similar tables from both databases"""
    
    matching_datasets = []
    dataset_sources = []
    
    # Find datasets that match the type (e.g., 'DOCUMENTHEADER')
    for name, df in datasets.items():
        if dataset_type.lower() in name.lower():
            # Add source database column
            df_with_source = df.copy()
            source_db = 'choice' if 'choice' in name.lower() else 'agency'
            df_with_source['source_database'] = source_db
            df_with_source['source_table'] = name
            
            matching_datasets.append(df_with_source)
            dataset_sources.append(name)
    
    if len(matching_datasets) == 0:
        print(f"   No datasets found for type: {dataset_type}")
        return None
    
    print(f"   Unifying {len(matching_datasets)} datasets: {', '.join(dataset_sources)}")
    
    try:
        # Find common columns across all datasets
        common_columns = set(matching_datasets[0].columns)
        for df in matching_datasets[1:]:
            common_columns = common_columns.intersection(set(df.columns))
        
        common_columns = list(common_columns)
        print(f"   Found {len(common_columns)} common columns")
        
        # Combine datasets using common columns
        unified_dfs = []
        for df in matching_datasets:
            unified_dfs.append(df[common_columns])
        
        unified_df = pd.concat(unified_dfs, ignore_index=True, sort=False)
        
        # Apply ETL processing to unified dataset
        unified_df = etl_pipeline.process_dataset(unified_df, f"unified_{dataset_type}")
        
        print(f"  ✅ Created unified dataset: {unified_df.shape[0]} rows × {unified_df.shape[1]} columns")
        
        return unified_df
        
    except Exception as e:
        print(f"  ❌ Error creating unified dataset for {dataset_type}: {e}")
        return None

# Create unified datasets
unified_datasets = {}

if processed_datasets:
    print(" Creating Unified Datasets for Cross-Database Analysis")
    print("=" * 55)
    
    # Define dataset types to unify
    unify_types = [
        'DOCUMENTHEADER',
        'DONATION_HEADER', 
        'RW_ORG'
    ]
    
    for dataset_type in unify_types:
        print(f"\n Creating unified {dataset_type} dataset...")
        
        unified_df = create_unified_dataset(processed_datasets, dataset_type, etl_pipeline)
        if unified_df is not None:
            unified_datasets[f"unified_{dataset_type.lower()}"] = unified_df
            
            # Save unified dataset
            etl_pipeline.save_dataset(unified_df, f"unified_{dataset_type.lower()}")
    
    print(f"\n✅ Created {len(unified_datasets)} unified datasets")
else:
    print(" No processed datasets available for unification")

# Display summary of all processed data
all_datasets = {**processed_datasets, **unified_datasets}

if all_datasets:
    print(f"\n ETL Pipeline Results Summary")
    print("=" * 40)
    print(f" Individual Datasets: {len(processed_datasets)}")
    print(f" Unified Datasets: {len(unified_datasets)}")
    print(f" Total Datasets Available: {len(all_datasets)}")
    
    print(f"\n Dataset Inventory:")
    total_records = 0
    for name, df in all_datasets.items():
        rows, cols = df.shape
        total_records += rows
        dataset_type = " Unified" if name.startswith('unified_') else " Individual"
        print(f"  {dataset_type:<12} {name:<35} {rows:>6,} rows × {cols:>3} columns")
    
    print(f"\n Total processed records: {total_records:,}")
else:
    print("❌ No datasets available after ETL processing")

 Creating Unified Datasets for Cross-Database Analysis

 Creating unified DOCUMENTHEADER dataset...
   Unifying 2 datasets: choice_DOCUMENTHEADER, agency_DOCUMENTHEADER
   Found 16 common columns

 Processing unified_DOCUMENTHEADER...
   Input: 1000 rows × 16 columns
   Column Standardization (unified_DOCUMENTHEADER): Standardized 0/16 column names
   Missing Values Cleanup (unified_DOCUMENTHEADER): Shape changed from (1000, 16) to (1000, 16)
   Data Type Standardization (unified_DOCUMENTHEADER): Converted 0 column types
   Duplicate Check (unified_DOCUMENTHEADER): No duplicates found
  ✅ Output: 1000 rows × 16 columns
  ✅ Created unified dataset: 1000 rows × 16 columns
   Data Export (unified_documentheader): Saved in 2 formats

 Creating unified DONATION_HEADER dataset...
   Unifying 1 datasets: choice_AMX_DONATION_HEADER
   Found 27 common columns

 Processing unified_DONATION_HEADER...
   Input: 500 rows × 27 columns
   Column Standardization (unified_DONATION_HEADER): Standardized

In [59]:
# Generate comprehensive ETL report and data quality summary
etl_report = etl_pipeline.generate_processing_report()

print("\n ETL Processing Report Summary")
print("=" * 40)
print(f"⏰ Processing completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f" Total transformation steps: {etl_report['total_steps']}")

# Data quality metrics
if all_datasets:
    print(f"\n Data Quality Metrics:")
    print("-" * 30)
    
    for dataset_name, df in all_datasets.items():
        if len(df) > 0:
            missing_pct = (df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100
            duplicate_pct = ((len(df) - len(df.drop_duplicates())) / len(df)) * 100
            
            print(f"   {dataset_name[:30]:<30}")
            print(f"       Missing data: {missing_pct:.1f}%")
            print(f"       Duplicates: {duplicate_pct:.1f}%")
            print(f"       Data types: {len(df.dtypes.unique())} unique types")
            print()

# Save final summary
summary = {
    'etl_completion_time': datetime.now().isoformat(),
    'datasets_processed': len(processed_datasets),
    'unified_datasets_created': len(unified_datasets),
    'total_records_processed': sum(len(df) for df in all_datasets.values()) if all_datasets else 0,
    'output_directory': str(output_path),
    'transformation_steps': etl_report['total_steps']
}

summary_file = output_path / 'etl_summary.json'
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

print(f" ETL summary saved to: {summary_file}")
print(f"\n ETL Pipeline Complete! Ready for Exploratory Data Analysis.")


 ETL Processing Report saved to: notebook_output/etl_processing_report.json

 ETL Processing Report Summary
⏰ Processing completed at: 2025-08-09 16:56:34
 Total transformation steps: 59

 Data Quality Metrics:
------------------------------
   choice_DOCUMENTHEADER         
       Missing data: 0.0%
       Duplicates: 0.0%
       Data types: 4 unique types

   choice_AMX_DONATION_HEADER    
       Missing data: 0.4%
       Duplicates: 0.0%
       Data types: 2 unique types

   choice_AMX_DONATION_LINES     
       Missing data: 5.3%
       Duplicates: 0.0%
       Data types: 3 unique types

   choice_RW_ORG                 
       Missing data: 3.7%
       Duplicates: 0.0%
       Data types: 4 unique types

   agency_DOCUMENTHEADER         
       Missing data: 0.0%
       Duplicates: 0.0%
       Data types: 4 unique types

   agency_DOCUMENTLINES          
       Missing data: 0.0%
       Duplicates: 0.0%
       Data types: 4 unique types

   agency_RW_ORG                 
       Mi

## 4. Exploratory Data Analysis (EDA)

Comprehensive statistical analysis of the processed HungerHub data to understand patterns, relationships, and insights.

### EDA Strategy:
1. **Univariate Analysis**: Individual variable distributions and statistics
2. **Bivariate Analysis**: Relationships between pairs of variables
3. **Multivariate Analysis**: Complex relationships across multiple variables
4. **Visual Analysis**: Interactive charts and plots using Plotly Express
5. **Statistical Testing**: Hypothesis testing and correlation analysis

### Key Questions to Answer:
- What are the distributions of key metrics (donations, orders, organizations)?
- How do food assistance patterns vary by geography and time?
- What relationships exist between donor behavior and recipient needs?
- Are there seasonal or cyclical patterns in the data?
- What anomalies or outliers require attention?

In [60]:
# EDA Helper Functions and Setup
class HungerHubEDA:
    """Exploratory Data Analysis toolkit for HungerHub data"""
    
    def __init__(self, datasets, output_dir):
        self.datasets = datasets
        self.output_dir = Path(output_dir)
        self.viz_dir = self.output_dir / 'visualizations'
        self.viz_dir.mkdir(exist_ok=True)
        
        self.insights = []
        self.summary_stats = {}
    
    def add_insight(self, category, insight):
        """Add an analytical insight"""
        self.insights.append({
            'category': category,
            'insight': insight,
            'timestamp': datetime.now().isoformat()
        })
        print(f" {category}: {insight}")
    
    def get_dataset_overview(self):
        """Get comprehensive overview of all datasets"""
        overview = {}
        
        print(" Dataset Overview Analysis")
        print("=" * 40)
        
        for name, df in self.datasets.items():
            if len(df) > 0:
                stats = {
                    'rows': len(df),
                    'columns': len(df.columns),
                    'memory_mb': df.memory_usage(deep=True).sum() / 1024**2,
                    'missing_cells': df.isnull().sum().sum(),
                    'missing_percentage': (df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100,
                    'numeric_columns': len(df.select_dtypes(include=[np.number]).columns),
                    'categorical_columns': len(df.select_dtypes(include=['object']).columns),
                    'datetime_columns': len(df.select_dtypes(include=['datetime64']).columns)
                }
                
                overview[name] = stats
                
                print(f"\n {name}:")
                print(f"   Dimensions: {stats['rows']:,} rows × {stats['columns']} columns")
                print(f"   Memory usage: {stats['memory_mb']:.1f} MB")
                print(f"   Missing data: {stats['missing_cells']:,} cells ({stats['missing_percentage']:.1f}%)")
                print(f"   Data types: {stats['numeric_columns']} numeric, {stats['categorical_columns']} categorical, {stats['datetime_columns']} datetime")
                
                # Add insights based on the data characteristics
                if stats['missing_percentage'] > 20:
                    self.add_insight('Data Quality', f"{name} has {stats['missing_percentage']:.1f}% missing data - requires careful handling")
                
                if stats['rows'] > 10000:
                    self.add_insight('Scale', f"{name} is a large dataset with {stats['rows']:,} records - suitable for robust analysis")
        
        self.summary_stats['overview'] = overview
        return overview
    
    def analyze_numeric_columns(self, df, dataset_name, max_columns=10):
        """Comprehensive analysis of numeric columns"""
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        
        if len(numeric_cols) == 0:
            print(f"   No numeric columns found in {dataset_name}")
            return None
        
        # Limit analysis to prevent overwhelming output
        cols_to_analyze = numeric_cols[:max_columns]
        
        print(f"\n Numeric Analysis: {dataset_name} ({len(cols_to_analyze)} columns)")
        print("-" * 50)
        
        numeric_summary = df[cols_to_analyze].describe()
        
        # Enhanced statistics
        for col in cols_to_analyze:
            if df[col].count() > 0:  # Only analyze non-empty columns
                series = df[col].dropna()
                
                stats = {
                    'count': len(series),
                    'mean': series.mean(),
                    'median': series.median(),
                    'std': series.std(),
                    'min': series.min(),
                    'max': series.max(),
                    'skewness': series.skew() if len(series) > 1 else 0,
                    'zeros': (series == 0).sum(),
                    'unique_values': series.nunique()
                }
                
                print(f"\n   {col}:")
                print(f"     Count: {stats['count']:,} | Unique: {stats['unique_values']:,} | Zeros: {stats['zeros']:,}")
                print(f"     Mean: {stats['mean']:.2f} | Median: {stats['median']:.2f} | Std: {stats['std']:.2f}")
                print(f"     Range: [{stats['min']:.2f}, {stats['max']:.2f}] | Skewness: {stats['skewness']:.2f}")
                
                # Generate insights
                if abs(stats['skewness']) > 2:
                    skew_type = 'highly right-skewed' if stats['skewness'] > 0 else 'highly left-skewed'
                    self.add_insight('Distribution', f"{col} in {dataset_name} is {skew_type} (skewness: {stats['skewness']:.2f})")
                
                if stats['zeros'] / stats['count'] > 0.5:
                    self.add_insight('Data Pattern', f"{col} in {dataset_name} has {(stats['zeros']/stats['count']*100):.1f}% zero values")
        
        return numeric_summary
    
    def analyze_categorical_columns(self, df, dataset_name, max_categories=20):
        """Analysis of categorical columns"""
        categorical_cols = df.select_dtypes(include=['object']).columns
        
        if len(categorical_cols) == 0:
            print(f"   No categorical columns found in {dataset_name}")
            return None
        
        print(f"\n Categorical Analysis: {dataset_name} ({len(categorical_cols)} columns)")
        print("-" * 50)
        
        categorical_summary = {}
        
        for col in categorical_cols[:10]:  # Limit to first 10 categorical columns
            if df[col].count() > 0:
                value_counts = df[col].value_counts()
                
                stats = {
                    'unique_count': df[col].nunique(),
                    'most_common': value_counts.index[0] if len(value_counts) > 0 else None,
                    'most_common_count': value_counts.iloc[0] if len(value_counts) > 0 else 0,
                    'top_categories': value_counts.head().to_dict()
                }
                
                categorical_summary[col] = stats
                
                print(f"\n   {col}:")
                print(f"     Unique values: {stats['unique_count']:,}")
                print(f"     Most common: '{stats['most_common']}' ({stats['most_common_count']:,} times)")
                
                if stats['unique_count'] <= max_categories:
                    print(f"     Top categories: {dict(list(stats['top_categories'].items())[:5])}")
                
                # Generate insights
                uniqueness_ratio = stats['unique_count'] / len(df)
                if uniqueness_ratio > 0.9:
                    self.add_insight('Data Quality', f"{col} in {dataset_name} has very high uniqueness ({uniqueness_ratio:.1%}) - possibly an identifier")
                elif stats['most_common_count'] / len(df) > 0.8:
                    dominance_pct = stats['most_common_count'] / len(df) * 100
                    self.add_insight('Data Pattern', f"{col} in {dataset_name} is dominated by '{stats['most_common']}' ({dominance_pct:.1f}% of records)")
        
        return categorical_summary

# Initialize EDA toolkit
if 'all_datasets' in globals() and all_datasets:
    eda = HungerHubEDA(all_datasets, notebook_outputs['base'])
    print("✅ EDA toolkit initialized with processed datasets")
    print(f" Ready to analyze {len(all_datasets)} datasets")
else:
    print(" No processed datasets available for EDA")
    print(" Please run Section 3 (ETL Pipeline) first to generate processed datasets")

✅ EDA toolkit initialized with processed datasets
 Ready to analyze 10 datasets


### Dataset Quick Inspection

Examining the structure and content of each processed dataset before detailed analysis.

In [61]:
# Quick inspection of all processed datasets
if 'all_datasets' in globals() and all_datasets:
    print("Dataset Quick Inspection")
    print("=" * 40)
    print(f"Total datasets available: {len(all_datasets)}")
    print()
    
    for dataset_name in all_datasets.keys():
        print(f"Dataset: {dataset_name}")
        print(f"Shape: {all_datasets[dataset_name].shape}")
        print()
else:
    print("No datasets available for inspection")
    print("Please run Section 3 (ETL Pipeline) first")

Dataset Quick Inspection
Total datasets available: 10

Dataset: choice_DOCUMENTHEADER
Shape: (500, 46)

Dataset: choice_AMX_DONATION_HEADER
Shape: (500, 25)

Dataset: choice_AMX_DONATION_LINES
Shape: (500, 23)

Dataset: choice_RW_ORG
Shape: (500, 27)

Dataset: agency_DOCUMENTHEADER
Shape: (500, 19)

Dataset: agency_DOCUMENTLINES
Shape: (500, 16)

Dataset: agency_RW_ORG
Shape: (500, 28)

Dataset: unified_documentheader
Shape: (1000, 16)

Dataset: unified_donation_header
Shape: (500, 27)

Dataset: unified_rw_org
Shape: (1000, 28)



In [62]:
# Dataset inspection will be generated dynamically
if 'all_datasets' in globals() and all_datasets:
    for dataset_name, df in all_datasets.items():
        print(f"\n{'='*60}")
        print(f"DATASET: {dataset_name.upper()}")
        print(f"{'='*60}")
        
        print(f"\n1. FIRST 10 ROWS:")
        print("-" * 30)
        display(df.head(10))
        
        print(f"\n2. DATASET INFO:")
        print("-" * 30)
        df.info()
        
        print(f"\n3. DESCRIPTIVE STATISTICS (ALL COLUMNS):")
        print("-" * 30)
        try:
            desc_stats = df.describe(include='all').T
            display(desc_stats)
        except Exception as e:
            print(f"Error generating descriptive statistics: {e}")
            # Fallback to basic describe
            try:
                display(df.describe().T)
            except:
                print("Unable to generate descriptive statistics for this dataset")
        
        print(f"\n" + "="*60 + "\n")
else:
    print("❌ No datasets available for detailed inspection")


DATASET: CHOICE_DOCUMENTHEADER

1. FIRST 10 ROWS:
------------------------------


,documentid,refnumber,sourceorg,userid,status,createddate,expirydate,prop1,prop2,prop3,prop4,prop5,prop6,prop7,prop8,prop9,prop10,prop11,prop12,prop13,prop15,notify,username,useremail,sourceorgname,domainid,domainname,prop16,prop18,prop19,prop20,prop21,prop22,prop23,prop24,prop25,prop26,prop27,prop28,prop29,prop30,prop31,prop32,prop33,prop34,prop35
0,L167066,L167066,ConAgraFoods,Unknown,3,1970-01-01 00:00:00.020220912,1970-01-01 00:00:00.020171107,Unknown,YD033,No,1,11/3/2017,Unknown,11/10/2017,Donor will deliver,Pallet Exchange,-2000,No,Truck Load,CSWANN,Ease on down the road Dan added shelf life and extenstion and storage as refrigerated for the 1...,202.0,Unknown,Unknown,ConAgra Consolidated,Unknown,Unknown,877-344-8070,cswann@feedingamerica.org,ConAgra,Kanesha Moss,1500 N Central Ave,Humboldt,TN,38343-1798,(859) 653-8989,Unknown,Kanesha.Moss@conagrafoods.com,0,No,-100,Unknown,Chicago,IL,60601,0
1,L167067,L167067,ConAgraFoods,Unknown,3,1970-01-01 00:00:00.020220912,1970-01-01 00:00:00.020171107,Unknown,YD033 (Load 2),No,1,11/3/2017,Unknown,11/10/2017,Donor will deliver,Pallet Exchange,-2000,No,Truck Load,CSWANN,Ease on down the road Dan added shelf life and extenstion and storage as refrigerated for the 1...,202.0,Unknown,Unknown,ConAgra Consolidated,Unknown,Unknown,877-344-8070,cswann@feedingamerica.org,ConAgra,Kanesha Moss,1500 N Central Ave,Humboldt,TN,38343-1798,(859) 653-8989,Unknown,Kanesha.Moss@conagrafoods.com,0,No,-33,Unknown,Chicago,IL,60601,0
2,ID_8709,ML8709,Chicago,USER_Z0007742,3,1970-01-01 00:00:00.020171106,1970-01-01 00:00:00.020171107,Chicago1,Soup jb,Yes,1,11/8/2017,C1234,11/15/2017,Member will pick up,Pallets - No Exchange,-100,No,Less Than Truck Load,SBARNES,Pick this load up ASAP please!,303.0,Mike Loeffl,mloeffl@feedingamerica.org,Greater Chicago Food Depository,DOM_00000001,DEFAULT,877-344-8070,sbarnes@feedingamerica.org,Greater Chicago Food Depository,Mike Loeffl,4100 West Ann Lurie Place,Chicago,IL,60632,(773) 247-3663,(773) 247-4232,mloeffl@chicago.org,0,Yes,1000,No Refers,Chicago,IL,60601,0
3,L167081,L167081,Kellogg,Unknown,3,1970-01-01 00:00:00.020220912,1970-01-01 00:00:00.020171107,Single LineChoice,AA YD022,No,2,10/10/2018,1,10/17/2018,Member will pick up,Slip Sheets,-2000,No,Truck Load,MLOEFFL,PO 498616 and 498162,202.0,Unknown,Unknown,Kellogg Company,Unknown,Unknown,877-344-8070,mloeffl@feedingamerica.org,Kellogg Montgomery DSD 2426,Gary Geisking,780 Industrial Park Blvd Ste F,Montgomery,AL,36117-5526,317-897-1726,Unknown,gary@hoosierwarehouses.com,0,No,-1800,FOR FULL TLs YOU WILL NEED A RELEASE NUMBER. YOU MUST CALL AT LEAST 48 HOURS AHEAD OF TIME TO M...,Chicago,IL,60601,0
4,L167082,L167082,Kellogg,Unknown,3,1970-01-01 00:00:00.020220912,1970-01-01 00:00:00.020171107,Single LineChoice,AA YD022 Rapid,Yes,1,10/10/2018,1,10/17/2018,Member will pick up,Slip Sheets,-2000,No,Truck Load,MLOEFFL,PO 498616 and 498162,202.0,Unknown,Unknown,Kellogg Company,Unknown,Unknown,877-344-8070,mloeffl@feedingamerica.org,Kellogg Montgomery DSD 2426,Gary Geisking,780 Industrial Park Blvd Ste F,Montgomery,AL,36117-5526,317-897-1726,Unknown,gary@hoosierwarehouses.com,0,No,200,FOR FULL TLs YOU WILL NEED A RELEASE NUMBER. YOU MUST CALL AT LEAST 48 HOURS AHEAD OF TIME TO M...,Chicago,IL,60601,0
5,L167113,L167113,SeaShare,Unknown,3,1970-01-01 00:00:00.020220912,1970-01-01 00:00:00.020171113,Unknown,AA SFL010,Yes,1,11/12/2017,Unknown,11/17/2017,Member will pick up,Arrange with Warehouse,-2000,No,Truck Load,MLOEFFL,This is from the special instructions line in the upload file,202.0,Unknown,Unknown,SEASHARE,Unknown,Unknown,877-344-8070,mloeffl@feedingamerica.org,SeaShare Fish Hut 1,Tom Noonen,256 Fish Head Lane,Green Bay,WI,54302,258-898-2235,Unknown,tn@seashare.org,0,No,-2000,Member must submit receipt (via Choice) within 48 hrs of pickup stating the NET WEIGHT. Any diff...,Chicago,IL,60601,0
6,L167283,L167283,SeaShare,Unknown,3,1970-01-01 00:00:00.020220912,1970-01-01 00:00:00.020171215,16 APA - 1,Dan seafood 12 14 1


2. DATASET INFO:
------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 46 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   documentid     500 non-null    object        
 1   refnumber      500 non-null    object        
 2   sourceorg      500 non-null    object        
 3   userid         500 non-null    object        
 4   status         500 non-null    int64         
 5   createddate    500 non-null    datetime64[ns]
 6   expirydate     500 non-null    datetime64[ns]
 7   prop1          500 non-null    object        
 8   prop2          500 non-null    object        
 9   prop3          500 non-null    object        
 10  prop4          500 non-null    object        
 11  prop5          500 non-null    object        
 12  prop6          500 non-null    object        
 13  prop7          500 non-null    object        
 14  prop8          500 non-nu

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
documentid,500,500,ML10289,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
refnumber,500,500,ML10289,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sourceorg,500,105,ConAgraFoods,52,NaN,NaN,NaN,NaN,NaN,NaN,NaN
userid,500,9,Unknown,440,NaN,NaN,NaN,NaN,NaN,NaN,NaN
status,500.0,NaN,NaN,NaN,6.444,-1.0,3.0,6.0,8.0,100.0,11.489599
createddate,500,NaN,NaN,NaN,1970-01-01 00:00:00.020219421,1970-01-01 00:00:00.020170823,1970-01-01 00:00:00.020220912,1970-01-01 00:00:00.020220912,1970-01-01 00:00:00.020220912,1970-01-01 00:00:00.020250710,NaN
expirydate,500,NaN,NaN,NaN,1970-01-01 00:00:00.020198372,1970-01-01 00:00:00.020170804,1970-01-01 00:00:00.020180311,1970-01-01 00:00:00.020181115,1970-01-01 00:00:00.020220210,1970-01-01 00:00:00.020250710,NaN
prop1,500,227,Unknown,195,NaN,NaN,NaN,NaN,NaN,NaN,NaN
prop2,500,402,Unknown,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN
prop3,500,3,No,291,NaN,NaN,NaN,NaN,NaN,NaN,NaN





DATASET: CHOICE_AMX_DONATION_HEADER

1. FIRST 10 ROWS:
------------------------------


...

,count,unique,top,freq,mean,min,25%,50%,75%,max
rcontactphone,500,176,Unknown,76,NaN,NaN,NaN,NaN,NaN,NaN
donorid,500,84,ConAgraFoods,63,NaN,NaN,NaN,NaN,NaN,NaN
donationdate,500,NaN,NaN,NaN,2018-04-30 20:47:02.400000256,1900-01-01 00:00:00,2018-03-12 00:00:00,2019-04-01 12:00:00,2022-04-26 06:00:00,2025-07-10 00:00:00
dcontactfax,500,30,Unknown,441,NaN,NaN,NaN,NaN,NaN,NaN
wcontactname,500,267,Tom Thimble,21,NaN,NaN,NaN,NaN,NaN,NaN
dcontactname,500,213,Mary Harmon,18,NaN,NaN,NaN,NaN,NaN,NaN
dcontactphone,500,202,Unknown,58,NaN,NaN,NaN,NaN,NaN,NaN
donorname,500,84,ConAgraFoods,63,NaN,NaN,NaN,NaN,NaN,NaN
rcontactname,500,183,Unknown,38,NaN,NaN,NaN,NaN,NaN,NaN
shipmentmethod,500,6,PICKUP,419,NaN,NaN,NaN,NaN,NaN,NaN





DATASET: UNIFIED_RW_ORG

1. FIRST 10 ROWS:
------------------------------


,entity_name,parent_org_id,annual_rev_vid,org_id,printable_name,source_table,currency_pattern,path,source_database,currency,created_time,master_org_id,modified_by,industry_val_num,float_pattern,sys_flags,country,org_type,years_in_business,modified_time,group_size,status_flag,language,oca,decimal_char,created_by,group_separator,emp_count_vid
0,SanLeandro,ORG_00004IFK,0.0,ORG_00004KEF,San Leandro,choice_RW_ORG,"*###,##0.00",ORG_00000001.ORG_00004IFH.ORG_00004IFK.ORG_00004KEF,choice,USD,2003-05-06 05:09:58,ORG_00004IFH,USER_00004K1T,0.0,"###,##0.###",0,US,S,0.0,2003-05-06 05:10:49,3,A,en,1,.,USER_00004K1T,",",0.0
1,WinchesterPlant,ORG_00004IFK,0.0,ORG_00004KH5,WinchesterPlant,choice_RW_ORG,"*###,##0.00",ORG_00000001.ORG_00004IFH.ORG_00004IFK.ORG_00004KH5,choice,USD,2003-05-09 03:01:28,ORG_00004IFH,USER_00004K1T,0.0,"###,##0.###",0,US,S,0.0,2003-05-09 03:02:31,3,A,en,1,.,USER_00004K1T,",",0.0
2,Albuquerque,ORG_00004IFI,0.0,ORG_00004KH6,Albuquerque,choice_RW_ORG,"*###,##0.00",ORG_00000001.ORG_00004IFI.ORG_00004KH6,choice,USD,2003-05-18 17:14:51,ORG_00004IFI,USER_00004SE9,0.0,"###,##0.###",0,US,S,0.0,2009-04-24 15:14:39,3,A,en,5,.,USER_00004K1T,",",0.0
3,MemphisZC,ORG_00004IFI,0.0,ORG_00004KH7,MemphisZC,choice_RW_ORG,"*###,##0.00",ORG_00000001.ORG_00004IFI.ORG_00004KH7,choice,USD,2003-05-18 17:29:09,ORG_00004IFI,USER_00004K1T,0.0,"###,##0.###",0,US,S,0.0,2003-05-18 17:31:33,3,A,en,2,.,USER_00004K1T,",",0.0
4,FtMeyersZC,ORG_00004IFI,0.0,ORG_00004KH8,FtMeyersZC,choice_RW_ORG,"*###,##0.00",ORG_00000001.ORG_00004IFI.ORG_00004KH8,choice,USD,2003-05-18 17:37:20,ORG_00004IFI,USER_00004Q9N,0.0,"###,##0.###",0,US,S,0.0,2006-01-05 00:27:12,3,A,en,5,.,USER_00004K1T,",",0.0
5,Fargo,ORG_00004IFI,0.0,ORG_00004KH9,Fargo,choice_RW_ORG,"*###,##0.00",ORG_00000001.ORG_00004IFI.ORG_00004KH9,choice,USD,2003-05-18 20:46:17,ORG_00004IFI,USER_00004Q9N,0.0,"###,##0.###",0,US,S,0.0,2006-01-05 00:19:25,3,A,en,3,.,USER_00004K1T,",",0.0
6,Memphis,ORG_00004IFI,0.0,ORG_00004KHA,Mid-South Food Bank,choice_RW_ORG,"*###,##0.00",ORG_00000001.ORG_00004IFI.ORG_00004KHA,choice,USD,2003-05-18 20:54:13,ORG_00004IFI,USER_00004SE9,0.0,"###,##0.###",0,US,S,0.0,2008-12-02 14:20:39,3,A,en,7,.,USER_00004K1T,",",0.0
7,Pennsauken,ORG_00004IFI,0.0,ORG_00004KHB,Pennsauken,choice_RW_ORG,"*###,##0.00",ORG_00000001.ORG_00004IFI.ORG_00004KHB,choice,USD,2003-05-18 20:57:48,ORG_00004IFI,USER_00004Q9N,0.0,"###,##0.###",0,US,S,0.0,2006-01-05 23:53:35,3,A,en,3,.,USER_00004K1T,",",0.0
8,MemphisZD,ORG_00004IFI,0.0,ORG_00004KHC,MemphisZD,choice_RW_ORG,"*###,##0.00",ORG_00000001.ORG_00004IFI.ORG_00004KHC,choice,USD,2003-05-19 12:27:45,ORG_00004IFI,USER_00004Q9N,0.0,"###,##0.###",0,US,S,0.0,2006-01-05 23:15:08,3,I,en,3,.,USER_00004K1T,",",0.0
9,TestAffiliate6,ORG_00004IFI,0.0,ORG_00004KK9,Test Affiliate6,choice_RW_ORG,"*###,##0.00",ORG_00000001.ORG_00004IFI.ORG_00004KK9,choice,USD,2003-05-22 20:21:44,ORG_00004IFI,USER_00004K1T,0.0,"###,##0.###",0,US,S,0.0,2003-05-22 20:22:44,3,A,en,2,.,USER_00004K1T,",",0.0



2. DATASET INFO:
------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 28 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   entity_name        1000 non-null   object        
 1   parent_org_id      1000 non-null   object        
 2   annual_rev_vid     1000 non-null   object        
 3   org_id             1000 non-null   object        
 4   printable_name     1000 non-null   object        
 5   source_table       1000 non-null   object        
 6   currency_pattern   1000 non-null   object        
 7   path               1000 non-null   object        
 8   source_database    1000 non-null   object        
 9   currency           1000 non-null   object        
 10  created_time       1000 non-null   datetime64[ns]
 11  master_org_id      1000 non-null   object        
 12  modified_by        1000 non-null   object        
 13  industry_val_nu

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
entity_name,1000,999,DEFAULT,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
parent_org_id,1000,246,ORG_00004J5D,259,NaN,NaN,NaN,NaN,NaN,NaN,NaN
annual_rev_vid,1000,1,0.0,1000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
org_id,1000,929,ORG_00004K4Q,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
printable_name,1000,858,Unknown,124,NaN,NaN,NaN,NaN,NaN,NaN,NaN
source_table,1000,2,choice_RW_ORG,500,NaN,NaN,NaN,NaN,NaN,NaN,NaN
currency_pattern,1000,1,"*###,##0.00",1000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
path,1000,995,ORG_00000001.ORG_00004IIL,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
source_database,1000,2,choice,500,NaN,NaN,NaN,NaN,NaN,NaN,NaN
currency,1000,1,USD,1000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 4.1 Univariate Analysis - Individual Variable Distributions

Analyzing each variable independently to understand distributions, central tendencies, and outliers.

In [63]:
# Comprehensive Univariate Analysis
if 'eda' in globals():
    print(" UNIVARIATE ANALYSIS")
    print("=" * 50)
    
    # Dataset Overview
    overview = eda.get_dataset_overview()
    
    # Analyze each dataset
    univariate_results = {}
    
    print(f" Analyzing datasets... ({len(eda.datasets)} total)\n")
    for dataset_name, df in eda.datasets.items():
        if len(df) > 0:
            print(f" Processing {dataset_name}...\n")
            print(f"\n{'='*60}")
            print(f" DETAILED ANALYSIS: {dataset_name.upper()}")
            print(f"{'='*60}")
            
            # Numeric analysis
            numeric_summary = eda.analyze_numeric_columns(df, dataset_name)
            
            # Categorical analysis 
            categorical_summary = eda.analyze_categorical_columns(df, dataset_name)
            
            # Store results
            univariate_results[dataset_name] = {
                'numeric_summary': numeric_summary,
                'categorical_summary': categorical_summary
            }
            
            # Look for datetime columns
            datetime_cols = df.select_dtypes(include=['datetime64']).columns
            if len(datetime_cols) > 0:
                print(f"\n DateTime Analysis: {len(datetime_cols)} columns")
                for col in datetime_cols[:3]:  # Limit to first 3 datetime columns
                    if df[col].count() > 0:
                        date_range = df[col].dropna()
                        print(f"   {col}: {date_range.min()} to {date_range.max()} ({len(date_range)} records)")
                        
                        # Check for date patterns
                        date_span_days = (date_range.max() - date_range.min()).days
                        if date_span_days > 365:
                            eda.add_insight('Temporal Coverage', f"{col} in {dataset_name} spans {date_span_days} days - good for trend analysis")

    print(f"\n✅ Univariate analysis completed for {len(univariate_results)} datasets")
else:
    print(" EDA toolkit not available - run previous section first")

 UNIVARIATE ANALYSIS
 Dataset Overview Analysis

 choice_DOCUMENTHEADER:
   Dimensions: 500 rows × 46 columns
   Memory usage: 1.3 MB
   Missing data: 0 cells (0.0%)
   Data types: 2 numeric, 42 categorical, 2 datetime

 choice_AMX_DONATION_HEADER:
   Dimensions: 500 rows × 25 columns
   Memory usage: 0.7 MB
   Missing data: 51 cells (0.4%)
   Data types: 0 numeric, 22 categorical, 3 datetime

 choice_AMX_DONATION_LINES:
   Dimensions: 500 rows × 23 columns
   Memory usage: 0.4 MB
   Missing data: 609 cells (5.3%)
   Data types: 6 numeric, 14 categorical, 3 datetime

 choice_RW_ORG:
   Dimensions: 500 rows × 27 columns
   Memory usage: 0.6 MB
   Missing data: 500 cells (3.7%)
   Data types: 5 numeric, 19 categorical, 3 datetime

 agency_DOCUMENTHEADER:
   Dimensions: 500 rows × 19 columns
   Memory usage: 0.3 MB
   Missing data: 0 cells (0.0%)
   Data types: 4 numeric, 11 categorical, 4 datetime

 agency_DOCUMENTLINES:
   Dimensions: 500 rows × 16 columns
   Memory usage: 0.3 MB
   Mis

### 4.2 Visual Analysis - Distribution and Pattern Visualization

Creating interactive visualizations to understand data distributions and identify patterns.

In [64]:
# Visualization Functions for EDA
def create_distribution_plots(df, dataset_name, numeric_cols, max_plots=6):
    """Create distribution plots for numeric columns"""
    
    if len(numeric_cols) == 0:
        print(f"   No numeric columns to plot for {dataset_name}")
        return None
    
    # Limit number of plots
    cols_to_plot = numeric_cols[:max_plots]
    
    # Calculate subplot layout
    n_plots = len(cols_to_plot)
    n_cols = min(3, n_plots)
    n_rows = (n_plots + n_cols - 1) // n_cols
    
    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=[f"{col} Distribution" for col in cols_to_plot],
        vertical_spacing=0.12
    )
    
    for i, col in enumerate(cols_to_plot):
        row = (i // n_cols) + 1
        col_pos = (i % n_cols) + 1
        
        # Clean data for plotting
        data = df[col].dropna()
        
        if len(data) > 0:
            fig.add_trace(
                go.Histogram(
                    x=data,
                    nbinsx=30,
                    name=col,
                    showlegend=False,
                    opacity=0.7
                ),
                row=row, col=col_pos
            )
    
    fig.update_layout(
        height=300 * n_rows,
        title_text=f" Numeric Distributions: {dataset_name}",
        showlegend=False
    )
    
    return fig

def create_category_plots(df, dataset_name, categorical_cols, max_plots=4):
    """Create bar plots for top categorical values"""
    
    if len(categorical_cols) == 0:
        print(f"   No categorical columns to plot for {dataset_name}")
        return None
    
    cols_to_plot = categorical_cols[:max_plots]
    
    # Calculate subplot layout
    n_plots = len(cols_to_plot)
    n_cols = min(2, n_plots)
    n_rows = (n_plots + n_cols - 1) // n_cols
    
    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=[f"Top {col} Values" for col in cols_to_plot],
        vertical_spacing=0.15
    )
    
    for i, col in enumerate(cols_to_plot):
        row = (i // n_cols) + 1
        col_pos = (i % n_cols) + 1
        
        # Get top 10 values
        top_values = df[col].value_counts().head(10)
        
        if len(top_values) > 0:
            fig.add_trace(
                go.Bar(
                    x=top_values.values,
                    y=top_values.index,
                    orientation='h',
                    name=col,
                    showlegend=False
                ),
                row=row, col=col_pos
            )
    
    fig.update_layout(
        height=400 * n_rows,
        title_text=f" Categorical Distributions: {dataset_name}",
        showlegend=False
    )
    
    return fig

# Generate visualizations for key datasets
if 'eda' in globals():
    print(" CREATING DISTRIBUTION VISUALIZATIONS")
    print("=" * 50)
    
    visualization_results = {}
    
    # Focus on the most important datasets (limit to prevent overwhelming output)
    priority_datasets = list(eda.datasets.keys())[:4]  # First 4 datasets
    
    for dataset_name in priority_datasets:
        df = eda.datasets[dataset_name]
        
        if len(df) > 0:
            print(f"\n Visualizing {dataset_name}...")
            
            # Get column types
            numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
            categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
            
            results = {}
            
            # Create numeric distribution plots
            if len(numeric_cols) > 0:
                print(f"   Creating {len(numeric_cols[:6])} numeric distribution plots...")
                dist_fig = create_distribution_plots(df, dataset_name, numeric_cols)
                if dist_fig:
                    dist_fig.show()
                    
                    # Save plot
                    plot_file = eda.viz_dir / f"{dataset_name}_distributions.html"
                    dist_fig.write_html(plot_file)
                    print(f"     Saved: {plot_file.name}")
                    results['distributions'] = str(plot_file)
            
            # Create categorical plots
            if len(categorical_cols) > 0:
                print(f"   Creating {len(categorical_cols[:4])} categorical plots...")
                cat_fig = create_category_plots(df, dataset_name, categorical_cols)
                if cat_fig:
                    cat_fig.show()
                    
                    # Save plot
                    plot_file = eda.viz_dir / f"{dataset_name}_categories.html"
                    cat_fig.write_html(plot_file)
                    print(f"     Saved: {plot_file.name}")
                    results['categories'] = str(plot_file)
            
            visualization_results[dataset_name] = results
    
    print(f"\n✅ Created visualizations for {len(visualization_results)} datasets")
    print(f" All plots saved to: {eda.viz_dir}")
else:
    print(" EDA toolkit not available - run previous sections first")

 CREATING DISTRIBUTION VISUALIZATIONS

 Visualizing choice_DOCUMENTHEADER...
   Creating 2 numeric distribution plots...


     Saved: choice_DOCUMENTHEADER_distributions.html
   Creating 4 categorical plots...


     Saved: choice_DOCUMENTHEADER_categories.html

 Visualizing choice_AMX_DONATION_HEADER...
   Creating 4 categorical plots...


     Saved: choice_AMX_DONATION_HEADER_categories.html

 Visualizing choice_AMX_DONATION_LINES...
   Creating 6 numeric distribution plots...


     Saved: choice_AMX_DONATION_LINES_distributions.html
   Creating 4 categorical plots...


     Saved: choice_AMX_DONATION_LINES_categories.html

 Visualizing choice_RW_ORG...
   Creating 5 numeric distribution plots...


     Saved: choice_RW_ORG_distributions.html
   Creating 4 categorical plots...


     Saved: choice_RW_ORG_categories.html

✅ Created visualizations for 4 datasets
 All plots saved to: notebook_output/visualizations


## 5. Bivariate & Multivariate Analysis

Analyzing relationships between variables to understand dependencies, correlations, and complex patterns in the HungerHub data.

### Analysis Strategy:
1. **Bivariate Analysis**: Relationships between pairs of variables
   - Numeric vs Numeric: Correlation analysis and scatter plots
   - Numeric vs Categorical: Group comparisons and box plots
   - Categorical vs Categorical: Cross-tabulations and chi-square tests

2. **Multivariate Analysis**: Complex relationships across multiple variables
   - Correlation matrices with hierarchical clustering
   - Principal Component Analysis (PCA) for dimensionality reduction
   - Multi-dimensional scatter plots and parallel coordinates

### Key Questions:
- Which variables are strongly correlated?
- How do donations vary by organization type and geography?
- What are the key drivers of food distribution volumes?
- Are there hidden patterns or clusters in the data?

In [65]:
# Advanced Analysis Functions
class HungerHubAdvancedAnalysis:
    """Advanced bivariate and multivariate analysis toolkit"""
    
    def __init__(self, datasets, output_dir):
        self.datasets = datasets
        self.output_dir = Path(output_dir)
        self.viz_dir = self.output_dir / 'visualizations'
        self.viz_dir.mkdir(exist_ok=True)
        
        self.correlations = {}
        self.insights = []
    
    def add_insight(self, category, insight, dataset=None):
        """Add analytical insight"""
        entry = {
            'category': category,
            'insight': insight,
            'dataset': dataset,
            'timestamp': datetime.now().isoformat()
        }
        self.insights.append(entry)
        dataset_info = f" ({dataset})" if dataset else ""
        print(f"Insight - {category}{dataset_info}: {insight}")
    
    def calculate_correlations(self, df, dataset_name, min_correlation=0.3):
        """Calculate and analyze correlations between numeric variables"""
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        
        if len(numeric_cols) < 2:
            print(f"  Insufficient numeric columns ({len(numeric_cols)}) for correlation analysis in {dataset_name}")
            return None
        
        # Calculate correlation matrix
        corr_matrix = df[numeric_cols].corr()
        
        # Find strong correlations
        strong_correlations = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                col1, col2 = corr_matrix.columns[i], corr_matrix.columns[j]
                correlation = corr_matrix.iloc[i, j]
                
                if not np.isnan(correlation) and abs(correlation) >= min_correlation:
                    strong_correlations.append({
                        'var1': col1,
                        'var2': col2,
                        'correlation': correlation,
                        'strength': 'Strong' if abs(correlation) > 0.7 else 'Moderate'
                    })
        
        # Sort by absolute correlation strength
        strong_correlations.sort(key=lambda x: abs(x['correlation']), reverse=True)
        
        print(f"\nCorrelation Analysis: {dataset_name}")
        print("-" * 50)
        print(f"Numeric columns analyzed: {len(numeric_cols)}")
        print(f"Strong correlations found (|r| >= {min_correlation}): {len(strong_correlations)}")
        
        if strong_correlations:
            print("\nTop correlations:")
            for corr in strong_correlations[:10]:  # Show top 10
                print(f"  {corr['var1']} <-> {corr['var2']}: {corr['correlation']:.3f} ({corr['strength']})")
                
                # Generate insights
                if abs(corr['correlation']) > 0.8:
                    self.add_insight('Strong Correlation', 
                                   f"{corr['var1']} and {corr['var2']} are highly correlated (r={corr['correlation']:.3f})",
                                   dataset_name)
        
        self.correlations[dataset_name] = {
            'matrix': corr_matrix,
            'strong_correlations': strong_correlations
        }
        
        return corr_matrix
    
    def analyze_numeric_relationships(self, df, dataset_name, max_pairs=5):
        """Analyze relationships between numeric variables"""
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        
        if len(numeric_cols) < 2:
            return None
        
        # Get top correlated pairs from correlation analysis
        if dataset_name in self.correlations and self.correlations[dataset_name]['strong_correlations']:
            top_pairs = self.correlations[dataset_name]['strong_correlations'][:max_pairs]
            
            print(f"\nNumeric Relationship Analysis: {dataset_name}")
            print("-" * 50)
            
            for pair in top_pairs:
                var1, var2 = pair['var1'], pair['var2']
                correlation = pair['correlation']
                
                # Calculate additional statistics
                valid_data = df[[var1, var2]].dropna()
                if len(valid_data) > 10:  # Need sufficient data points
                    
                    # Linear regression statistics
                    x, y = valid_data[var1], valid_data[var2]
                    if x.std() > 0 and y.std() > 0:  # Avoid division by zero
                        
                        print(f"\n  {var1} vs {var2}:")
                        print(f"    Correlation: {correlation:.3f}")
                        print(f"    Sample size: {len(valid_data)}")
                        print(f"    {var1} range: [{x.min():.2f}, {x.max():.2f}]")
                        print(f"    {var2} range: [{y.min():.2f}, {y.max():.2f}]")
                        
                        # Check for outliers using IQR
                        def count_outliers(series):
                            Q1, Q3 = series.quantile([0.25, 0.75])
                            IQR = Q3 - Q1
                            lower_bound = Q1 - 1.5 * IQR
                            upper_bound = Q3 + 1.5 * IQR
                            return ((series < lower_bound) | (series > upper_bound)).sum()
                        
                        x_outliers = count_outliers(x)
                        y_outliers = count_outliers(y)
                        
                        if x_outliers > 0 or y_outliers > 0:
                            print(f"    Outliers: {x_outliers} in {var1}, {y_outliers} in {var2}")
                            
                            if (x_outliers + y_outliers) / len(valid_data) > 0.1:
                                self.add_insight('Data Quality',
                                               f"{var1} vs {var2} relationship has many outliers ({x_outliers + y_outliers} total)",
                                               dataset_name)
        
        return True
    
    def analyze_categorical_relationships(self, df, dataset_name, max_combinations=3):
        """Analyze relationships between categorical variables"""
        categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
        
        if len(categorical_cols) < 2:
            print(f"  Insufficient categorical columns ({len(categorical_cols)}) for relationship analysis in {dataset_name}")
            return None
        
        print(f"\nCategorical Relationship Analysis: {dataset_name}")
        print("-" * 50)
        
        # Analyze pairs of categorical variables
        combinations_analyzed = 0
        for i in range(len(categorical_cols)):
            for j in range(i+1, len(categorical_cols)):
                if combinations_analyzed >= max_combinations:
                    break
                    
                col1, col2 = categorical_cols[i], categorical_cols[j]
                
                # Filter to reasonable number of categories
                if df[col1].nunique() <= 20 and df[col2].nunique() <= 20:
                    crosstab = pd.crosstab(df[col1], df[col2])
                    
                    print(f"\n  {col1} vs {col2}:")
                    print(f"    {col1} categories: {df[col1].nunique()}")
                    print(f"    {col2} categories: {df[col2].nunique()}")
                    print(f"    Cross-tabulation shape: {crosstab.shape}")
                    
                    # Look for dominant patterns
                    total_combinations = crosstab.sum().sum()
                    max_cell = crosstab.max().max()
                    dominance_ratio = max_cell / total_combinations
                    
                    if dominance_ratio > 0.5:
                        max_idx = crosstab.stack().idxmax()
                        self.add_insight('Pattern',
                                       f"{col1}='{max_idx[0]}' and {col2}='{max_idx[1]}' combination dominates ({dominance_ratio:.1%} of data)",
                                       dataset_name)
                    
                    combinations_analyzed += 1
            
            if combinations_analyzed >= max_combinations:
                break
        
        return True

# Initialize advanced analysis
if 'all_datasets' in globals() and all_datasets:
    advanced_analysis = HungerHubAdvancedAnalysis(all_datasets, notebook_outputs['base'])
    print("✅ Advanced analysis toolkit initialized")
    print(f"Ready for bivariate and multivariate analysis of {len(all_datasets)} datasets")
else:
    print("Advanced analysis requires processed datasets")
    print("Please run previous sections first")

✅ Advanced analysis toolkit initialized
Ready for bivariate and multivariate analysis of 10 datasets


### 5.1 Correlation Analysis

Analyzing correlations between numeric variables to identify strong relationships and dependencies.

In [66]:
# Comprehensive Correlation Analysis
if 'advanced_analysis' in globals():
    print("CORRELATION ANALYSIS")
    print("=" * 50)
    
    correlation_results = {}
    
    # Analyze correlations for each dataset
    for dataset_name, df in advanced_analysis.datasets.items():
        if len(df) > 0:
            print(f"\nAnalyzing correlations in: {dataset_name}")
            
            corr_matrix = advanced_analysis.calculate_correlations(df, dataset_name)
            if corr_matrix is not None:
                correlation_results[dataset_name] = corr_matrix
    
    print(f"\n✅ Correlation analysis completed for {len(correlation_results)} datasets")
else:
    print("❌ Advanced analysis toolkit not available")

CORRELATION ANALYSIS

Analyzing correlations in: choice_DOCUMENTHEADER

Correlation Analysis: choice_DOCUMENTHEADER
--------------------------------------------------
Numeric columns analyzed: 2
Strong correlations found (|r| >= 0.3): 0

Analyzing correlations in: choice_AMX_DONATION_HEADER
  Insufficient numeric columns (0) for correlation analysis in choice_AMX_DONATION_HEADER

Analyzing correlations in: choice_AMX_DONATION_LINES

Correlation Analysis: choice_AMX_DONATION_LINES
--------------------------------------------------
Numeric columns analyzed: 6
Strong correlations found (|r| >= 0.3): 4

Top correlations:
  quantity <-> totalpallets: 0.823 (Strong)
Insight - Strong Correlation (choice_AMX_DONATION_LINES): quantity and totalpallets are highly correlated (r=0.823)
  totalpallets <-> totalgrossweight: 0.674 (Moderate)
  quantity <-> totalgrossweight: 0.609 (Moderate)
  grossweight <-> netweight: 0.538 (Moderate)

Analyzing correlations in: choice_RW_ORG

Correlation Analysis: 

### 5.2 Bivariate Relationship Analysis

Detailed analysis of relationships between pairs of variables.

In [67]:
# Bivariate Relationship Analysis
if 'advanced_analysis' in globals():
    print("BIVARIATE RELATIONSHIP ANALYSIS")
    print("=" * 50)
    
    # Analyze numeric relationships
    print("\n1. NUMERIC VARIABLE RELATIONSHIPS:")
    print("=" * 40)
    
    for dataset_name, df in advanced_analysis.datasets.items():
        if len(df) > 0:
            advanced_analysis.analyze_numeric_relationships(df, dataset_name)
    
    # Analyze categorical relationships
    print("\n\n2. CATEGORICAL VARIABLE RELATIONSHIPS:")
    print("=" * 40)
    
    for dataset_name, df in advanced_analysis.datasets.items():
        if len(df) > 0:
            advanced_analysis.analyze_categorical_relationships(df, dataset_name)
    
    print("\n✅ Bivariate analysis completed")
else:
    print("❌ Advanced analysis toolkit not available")

BIVARIATE RELATIONSHIP ANALYSIS

1. NUMERIC VARIABLE RELATIONSHIPS:

Numeric Relationship Analysis: choice_AMX_DONATION_LINES
--------------------------------------------------

  quantity vs totalpallets:
    Correlation: 0.823
    Sample size: 500
    quantity range: [1.00, 6001.00]
    totalpallets range: [0.00, 200.03]
    Outliers: 78 in quantity, 107 in totalpallets
Insight - Data Quality (choice_AMX_DONATION_LINES): quantity vs totalpallets relationship has many outliers (185 total)

  totalpallets vs totalgrossweight:
    Correlation: 0.674
    Sample size: 500
    totalpallets range: [0.00, 200.03]
    totalgrossweight range: [0.05, 60010.00]
    Outliers: 107 in totalpallets, 74 in totalgrossweight
Insight - Data Quality (choice_AMX_DONATION_LINES): totalpallets vs totalgrossweight relationship has many outliers (181 total)

  quantity vs totalgrossweight:
    Correlation: 0.609
    Sample size: 500
    quantity range: [1.00, 6001.00]
    totalgrossweight range: [0.05, 60010.

    Outliers: 78 in quantity, 74 in totalgrossweight
Insight - Data Quality (choice_AMX_DONATION_LINES): quantity vs totalgrossweight relationship has many outliers (152 total)

  grossweight vs netweight:
    Correlation: 0.538
    Sample size: 500
    grossweight range: [0.00, 2231.00]
    netweight range: [2.19, 2327.00]
    Outliers: 45 in grossweight, 41 in netweight
Insight - Data Quality (choice_AMX_DONATION_LINES): grossweight vs netweight relationship has many outliers (86 total)

Numeric Relationship Analysis: agency_DOCUMENTLINES
--------------------------------------------------

  lineitemnumber vs donatedquantity:
    Correlation: 0.319
    Sample size: 500
    lineitemnumber range: [1.00, 43.00]
    donatedquantity range: [0.00, 30.00]
    Outliers: 65 in lineitemnumber, 34 in donatedquantity
Insight - Data Quality (agency_DOCUMENTLINES): lineitemnumber vs donatedquantity relationship has many outliers (99 total)


2. CATEGORICAL VARIABLE RELATIONSHIPS:

Categorical Rela

### 5.3 Advanced Visualizations

Creating advanced visualizations to explore multivariate relationships and patterns.

In [68]:
# Advanced Visualization Functions
def create_correlation_heatmap(corr_matrix, dataset_name, min_size=3):
    """Create interactive correlation heatmap"""
    
    if corr_matrix is None or corr_matrix.shape[0] < min_size:
        print(f"  Insufficient data for correlation heatmap in {dataset_name}")
        return None
    
    fig = px.imshow(
        corr_matrix,
        x=corr_matrix.columns,
        y=corr_matrix.columns,
        color_continuous_scale='RdBu_r',
        range_color=[-1, 1],
        title=f"Correlation Matrix: {dataset_name}",
        aspect='auto'
    )
    
    fig.update_layout(
        height=max(400, len(corr_matrix.columns) * 25),
        width=max(400, len(corr_matrix.columns) * 25)
    )
    
    return fig

def create_scatter_matrix(df, dataset_name, numeric_cols, max_vars=5):
    """Create scatter plot matrix for numeric variables"""
    
    if len(numeric_cols) < 2:
        return None
    
    # Limit variables to prevent overwhelming plots
    cols_to_plot = numeric_cols[:max_vars]
    
    # Sample data if too large
    plot_df = df[cols_to_plot].dropna()
    if len(plot_df) > 1000:
        plot_df = plot_df.sample(n=1000, random_state=42)
    
    if len(plot_df) < 10:
        print(f"  Insufficient data points for scatter matrix in {dataset_name}")
        return None
    
    fig = px.scatter_matrix(
        plot_df,
        dimensions=cols_to_plot,
        title=f"Scatter Matrix: {dataset_name} (Top {len(cols_to_plot)} variables)",
        opacity=0.6
    )
    
    fig.update_traces(diagonal_visible=False)
    fig.update_layout(height=600, width=800)
    
    return fig

def create_grouped_analysis(df, dataset_name, numeric_col, categorical_col):
    """Create grouped analysis visualization"""
    
    # Check if columns exist and have reasonable cardinality
    if (numeric_col not in df.columns or categorical_col not in df.columns or
        df[categorical_col].nunique() > 20 or df[categorical_col].nunique() < 2):
        return None
    
    # Clean data
    clean_data = df[[numeric_col, categorical_col]].dropna()
    if len(clean_data) < 20:
        return None
    
    fig = px.box(
        clean_data,
        x=categorical_col,
        y=numeric_col,
        title=f"{numeric_col} by {categorical_col}: {dataset_name}",
        points="outliers"
    )
    
    fig.update_layout(
        height=400,
        xaxis_tickangle=45
    )
    
    return fig

# Generate advanced visualizations
if 'advanced_analysis' in globals() and 'correlation_results' in globals():
    print("ADVANCED VISUALIZATIONS")
    print("=" * 50)
    
    visualization_count = 0
    
    # Create correlation heatmaps
    print("\n1. CORRELATION HEATMAPS:")
    for dataset_name, corr_matrix in correlation_results.items():
        print(f"\nCreating correlation heatmap for {dataset_name}...")
        
        heatmap_fig = create_correlation_heatmap(corr_matrix, dataset_name)
        if heatmap_fig:
            heatmap_fig.show()
            
            # Save plot
            plot_file = advanced_analysis.viz_dir / f"{dataset_name}_correlation_heatmap.html"
            heatmap_fig.write_html(plot_file)
            print(f"  Saved: {plot_file.name}")
            visualization_count += 1
    
    # Create scatter matrices for datasets with sufficient numeric variables
    print("\n2. SCATTER MATRICES:")
    matrices_created = 0
    
    for dataset_name, df in advanced_analysis.datasets.items():
        if matrices_created >= 3:  # Limit to prevent too many plots
            break
            
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if len(numeric_cols) >= 3:
            print(f"\nCreating scatter matrix for {dataset_name}...")
            
            scatter_fig = create_scatter_matrix(df, dataset_name, numeric_cols)
            if scatter_fig:
                scatter_fig.show()
                
                # Save plot
                plot_file = advanced_analysis.viz_dir / f"{dataset_name}_scatter_matrix.html"
                scatter_fig.write_html(plot_file)
                print(f"  Saved: {plot_file.name}")
                visualization_count += 1
                matrices_created += 1
    
    # Create some grouped analysis examples
    print("\n3. GROUPED ANALYSIS EXAMPLES:")
    grouped_created = 0
    
    for dataset_name, df in advanced_analysis.datasets.items():
        if grouped_created >= 2:  # Limit examples
            break
            
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
        
        if len(numeric_cols) > 0 and len(categorical_cols) > 0:
            # Try first numeric and first categorical column
            numeric_col = numeric_cols[0]
            categorical_col = categorical_cols[0]
            
            print(f"\nCreating grouped analysis: {numeric_col} by {categorical_col} in {dataset_name}...")
            
            grouped_fig = create_grouped_analysis(df, dataset_name, numeric_col, categorical_col)
            if grouped_fig:
                grouped_fig.show()
                
                # Save plot
                plot_file = advanced_analysis.viz_dir / f"{dataset_name}_grouped_analysis.html"
                grouped_fig.write_html(plot_file)
                print(f"  Saved: {plot_file.name}")
                visualization_count += 1
                grouped_created += 1
    
    print(f"\n✅ Created {visualization_count} advanced visualizations")
    print(f"All plots saved to: {advanced_analysis.viz_dir}")
else:
    print("❌ Advanced analysis results not available")

ADVANCED VISUALIZATIONS

1. CORRELATION HEATMAPS:

Creating correlation heatmap for choice_DOCUMENTHEADER...
  Insufficient data for correlation heatmap in choice_DOCUMENTHEADER

Creating correlation heatmap for choice_AMX_DONATION_LINES...


  Saved: choice_AMX_DONATION_LINES_correlation_heatmap.html

Creating correlation heatmap for choice_RW_ORG...


  Saved: choice_RW_ORG_correlation_heatmap.html

Creating correlation heatmap for agency_DOCUMENTHEADER...


  Saved: agency_DOCUMENTHEADER_correlation_heatmap.html

Creating correlation heatmap for agency_DOCUMENTLINES...


  Saved: agency_DOCUMENTLINES_correlation_heatmap.html

Creating correlation heatmap for agency_RW_ORG...


...

  Saved: unified_rw_org_correlation_heatmap.html

2. SCATTER MATRICES:

Creating scatter matrix for choice_AMX_DONATION_LINES...


  Saved: choice_AMX_DONATION_LINES_scatter_matrix.html

Creating scatter matrix for choice_RW_ORG...


  Saved: choice_RW_ORG_scatter_matrix.html

Creating scatter matrix for agency_DOCUMENTHEADER...


  Saved: agency_DOCUMENTHEADER_scatter_matrix.html

3. GROUPED ANALYSIS EXAMPLES:

Creating grouped analysis: status by documentid in choice_DOCUMENTHEADER...

Creating grouped analysis: quantity by donationnumber in choice_AMX_DONATION_LINES...

Creating grouped analysis: group_size by org_id in choice_RW_ORG...

Creating grouped analysis: status by documentid in agency_DOCUMENTHEADER...

Creating grouped analysis: lineitemnumber by headerid in agency_DOCUMENTLINES...

Creating grouped analysis: group_size by org_id in agency_RW_ORG...

Creating grouped analysis: notify by domainid in unified_documentheader...


  Saved: unified_documentheader_grouped_analysis.html

Creating grouped analysis: industry_val_num by entity_name in unified_rw_org...

✅ Created 10 advanced visualizations
All plots saved to: notebook_output/visualizations


## 6. Business Intelligence Reports & Conclusions

Synthesizing analytical findings into actionable business intelligence reports and strategic recommendations for HungerHub operations.

### Report Strategy:
1. **Executive Summary**: Key findings and metrics overview
2. **Data Quality Assessment**: Comprehensive data health report
3. **Operational Insights**: Patterns affecting food distribution efficiency
4. **Strategic Recommendations**: Data-driven action items
5. **Technical Documentation**: Analysis methodology and limitations

### Business Questions Answered:
- What is the overall health and quality of HungerHub data?
- Which operational patterns drive the most efficient food distribution?
- What are the key performance indicators for hunger relief operations?
- Where should HungerHub focus improvement efforts?
- What data collection and analysis capabilities should be enhanced?

In [69]:
# Business Intelligence Report Generator
class HungerHubBusinessIntelligence:
    """Business intelligence and reporting toolkit"""
    
    def __init__(self, datasets, eda_results=None, advanced_results=None, output_dir=None):
        self.datasets = datasets
        self.eda_results = eda_results
        self.advanced_results = advanced_results
        self.output_dir = Path(output_dir) if output_dir else Path('notebook_output')
        self.reports_dir = self.output_dir / 'reports'
        self.reports_dir.mkdir(exist_ok=True)
        
        self.report_data = {
            'executive_summary': {},
            'data_quality': {},
            'operational_insights': [],
            'recommendations': [],
            'technical_details': {}
        }
    
    def generate_executive_summary(self):
        """Generate high-level executive summary"""
        print("EXECUTIVE SUMMARY")
        print("=" * 50)
        
        total_datasets = len(self.datasets)
        total_records = sum(len(df) for df in self.datasets.values())
        total_columns = sum(len(df.columns) for df in self.datasets.values())
        
        # Dataset size analysis
        dataset_sizes = [(name, len(df)) for name, df in self.datasets.items()]
        dataset_sizes.sort(key=lambda x: x[1], reverse=True)
        
        # Data type analysis
        numeric_cols = sum(len(df.select_dtypes(include=[np.number]).columns) for df in self.datasets.values())
        categorical_cols = sum(len(df.select_dtypes(include=['object']).columns) for df in self.datasets.values())
        datetime_cols = sum(len(df.select_dtypes(include=['datetime64']).columns) for df in self.datasets.values())
        
        summary = {
            'total_datasets': total_datasets,
            'total_records': total_records,
            'total_columns': total_columns,
            'largest_dataset': dataset_sizes[0] if dataset_sizes else ('None', 0),
            'numeric_columns': numeric_cols,
            'categorical_columns': categorical_cols,
            'datetime_columns': datetime_cols
        }
        
        print(f"\nDATA INVENTORY:")
        print(f"  Total Datasets Analyzed: {summary['total_datasets']}")
        print(f"  Total Records Processed: {summary['total_records']:,}")
        print(f"  Total Data Fields: {summary['total_columns']}")
        print(f"  Largest Dataset: {summary['largest_dataset'][0]} ({summary['largest_dataset'][1]:,} records)")
        
        print(f"\nDATA COMPOSITION:")
        print(f"  Numeric Fields: {summary['numeric_columns']} ({numeric_cols/total_columns*100:.1f}%)")
        print(f"  Categorical Fields: {summary['categorical_columns']} ({categorical_cols/total_columns*100:.1f}%)")
        print(f"  DateTime Fields: {summary['datetime_columns']} ({datetime_cols/total_columns*100:.1f}%)")
        
        print(f"\nTOP 5 DATASETS BY SIZE:")
        for i, (name, size) in enumerate(dataset_sizes[:5], 1):
            print(f"  {i}. {name}: {size:,} records")
        
        self.report_data['executive_summary'] = summary
        return summary
    
    def assess_data_quality(self):
        """Comprehensive data quality assessment"""
        print("\n\nDATA QUALITY ASSESSMENT")
        print("=" * 50)
        
        quality_report = {}
        overall_quality_score = 0
        total_assessments = 0
        
        for dataset_name, df in self.datasets.items():
            if len(df) == 0:
                continue
                
            print(f"\nAssessing: {dataset_name}")
            print("-" * 30)
            
            # Calculate quality metrics
            total_cells = len(df) * len(df.columns)
            missing_cells = df.isnull().sum().sum()
            missing_percentage = (missing_cells / total_cells) * 100 if total_cells > 0 else 0
            
            # Duplicate analysis
            duplicate_rows = len(df) - len(df.drop_duplicates())
            duplicate_percentage = (duplicate_rows / len(df)) * 100 if len(df) > 0 else 0
            
            # Data type consistency
            numeric_cols = df.select_dtypes(include=[np.number]).columns
            consistent_numerics = 0
            for col in numeric_cols:
                if df[col].dtype in ['int64', 'float64']:
                    consistent_numerics += 1
            
            type_consistency = (consistent_numerics / len(numeric_cols)) * 100 if len(numeric_cols) > 0 else 100
            
            # Calculate quality scores (0-100)
            completeness_score = max(0, 100 - missing_percentage)
            uniqueness_score = max(0, 100 - duplicate_percentage)
            consistency_score = type_consistency
            
            overall_score = (completeness_score + uniqueness_score + consistency_score) / 3
            
            quality_metrics = {
                'missing_percentage': missing_percentage,
                'duplicate_percentage': duplicate_percentage,
                'type_consistency': type_consistency,
                'completeness_score': completeness_score,
                'uniqueness_score': uniqueness_score,
                'consistency_score': consistency_score,
                'overall_score': overall_score
            }
            
            quality_report[dataset_name] = quality_metrics
            overall_quality_score += overall_score
            total_assessments += 1
            
            # Display results
            print(f"  Missing Data: {missing_percentage:.1f}% (Score: {completeness_score:.1f}/100)")
            print(f"  Duplicates: {duplicate_percentage:.1f}% (Score: {uniqueness_score:.1f}/100)")
            print(f"  Type Consistency: {type_consistency:.1f}% (Score: {consistency_score:.1f}/100)")
            print(f"  Overall Quality Score: {overall_score:.1f}/100")
            
            # Quality rating
            if overall_score >= 90:
                rating = "Excellent"
            elif overall_score >= 80:
                rating = "Good"
            elif overall_score >= 70:
                rating = "Fair"
            elif overall_score >= 60:
                rating = "Poor"
            else:
                rating = "Critical"
            
            print(f"  Quality Rating: {rating}")
        
        # Overall assessment
        avg_quality_score = overall_quality_score / total_assessments if total_assessments > 0 else 0
        
        print(f"\nOVERALL DATA QUALITY ASSESSMENT:")
        print("-" * 40)
        print(f"Average Quality Score: {avg_quality_score:.1f}/100")
        
        if avg_quality_score >= 85:
            overall_rating = "High Quality - Ready for production analytics"
        elif avg_quality_score >= 75:
            overall_rating = "Good Quality - Minor improvements recommended"
        elif avg_quality_score >= 65:
            overall_rating = "Fair Quality - Data cleaning required"
        else:
            overall_rating = "Low Quality - Significant data work needed"
        
        print(f"Overall Rating: {overall_rating}")
        
        self.report_data['data_quality'] = {
            'individual_scores': quality_report,
            'average_score': avg_quality_score,
            'overall_rating': overall_rating
        }
        
        return quality_report
    
    def generate_operational_insights(self):
        """Generate operational insights from analysis"""
        print("\n\nOPERATIONAL INSIGHTS")
        print("=" * 50)
        
        insights = []
        
        # Analyze dataset patterns
        print("\nDATA PATTERNS ANALYSIS:")
        print("-" * 30)
        
        # Find unified vs individual datasets
        unified_datasets = [name for name in self.datasets.keys() if 'unified' in name.lower()]
        individual_datasets = [name for name in self.datasets.keys() if 'unified' not in name.lower()]
        
        if unified_datasets:
            insight = f"Successfully created {len(unified_datasets)} unified datasets for cross-database analysis"
            insights.append(('Data Integration', insight))
            print(f"  Data Integration: {insight}")
        
        # Analyze data sources
        choice_datasets = [name for name in self.datasets.keys() if 'choice' in name.lower()]
        agency_datasets = [name for name in self.datasets.keys() if 'agency' in name.lower()]
        
        if choice_datasets and agency_datasets:
            insight = f"Multi-source data available: {len(choice_datasets)} Choice datasets, {len(agency_datasets)} Agency datasets"
            insights.append(('Data Coverage', insight))
            print(f"  Data Coverage: {insight}")
        
        # Analyze data volume distribution
        total_records = sum(len(df) for df in self.datasets.values())
        if total_records > 1000:
            insight = f"Substantial data volume ({total_records:,} records) suitable for statistical analysis and trend identification"
            insights.append(('Data Volume', insight))
            print(f"  Data Volume: {insight}")
        
        # Analyze data types for operational capabilities
        has_geographic = any('state' in str(df.columns).lower() or 'city' in str(df.columns).lower() 
                           or 'zip' in str(df.columns).lower() for df in self.datasets.values())
        has_temporal = any(len(df.select_dtypes(include=['datetime64']).columns) > 0 
                          for df in self.datasets.values())
        has_amounts = any('amount' in str(df.columns).lower() or 'quantity' in str(df.columns).lower()
                         or 'weight' in str(df.columns).lower() for df in self.datasets.values())
        
        if has_geographic:
            insight = "Geographic data available - enables location-based analysis and mapping"
            insights.append(('Geographic Analysis', insight))
            print(f"  Geographic Capability: {insight}")
        
        if has_temporal:
            insight = "Temporal data available - enables trend analysis and seasonal pattern identification"
            insights.append(('Temporal Analysis', insight))
            print(f"  Temporal Capability: {insight}")
        
        if has_amounts:
            insight = "Quantitative data available - enables volume and efficiency analysis"
            insights.append(('Quantitative Analysis', insight))
            print(f"  Quantitative Capability: {insight}")
        
        self.report_data['operational_insights'] = insights
        return insights
    
    def generate_recommendations(self):
        """Generate strategic recommendations based on analysis"""
        print("\n\nSTRATEGIC RECOMMENDATIONS")
        print("=" * 50)
        
        recommendations = []
        
        # Data quality recommendations
        if 'data_quality' in self.report_data:
            avg_score = self.report_data['data_quality']['average_score']
            
            print("\nDATA QUALITY IMPROVEMENTS:")
            print("-" * 30)
            
            if avg_score < 80:
                rec = "Implement data validation rules and quality monitoring dashboards"
                recommendations.append(('High Priority', 'Data Quality', rec))
                print(f"  High Priority: {rec}")
            
            if avg_score < 70:
                rec = "Establish data governance policies and regular data cleansing procedures"
                recommendations.append(('Critical', 'Data Governance', rec))
                print(f"  Critical: {rec}")
        
        # Analytics capabilities recommendations
        print("\nANALYTICS CAPABILITIES:")
        print("-" * 30)
        
        total_records = sum(len(df) for df in self.datasets.values())
        if total_records > 5000:
            rec = "Data volume supports advanced analytics - consider implementing machine learning models for predictive insights"
            recommendations.append(('Medium Priority', 'Advanced Analytics', rec))
            print(f"  Advanced Analytics: {rec}")
        
        # Integration recommendations
        unified_count = len([name for name in self.datasets.keys() if 'unified' in name.lower()])
        if unified_count > 0:
            rec = "Unified datasets created successfully - establish automated ETL pipelines for real-time data integration"
            recommendations.append(('Medium Priority', 'Data Integration', rec))
            print(f"  Data Integration: {rec}")
        
        # Visualization and reporting recommendations
        print("\nREPORTING & VISUALIZATION:")
        print("-" * 30)
        
        rec = "Deploy interactive dashboards for real-time operational monitoring"
        recommendations.append(('High Priority', 'Business Intelligence', rec))
        print(f"  Business Intelligence: {rec}")
        
        rec = "Implement automated report generation for key stakeholders"
        recommendations.append(('Medium Priority', 'Automation', rec))
        print(f"  Automation: {rec}")
        
        # Operational recommendations
        print("\nOPERATIONAL EFFICIENCY:")
        print("-" * 30)
        
        rec = "Leverage correlation analysis findings to optimize food distribution logistics"
        recommendations.append(('High Priority', 'Operations', rec))
        print(f"  Operations: {rec}")
        
        rec = "Use geographic and temporal patterns to improve resource allocation"
        recommendations.append(('High Priority', 'Resource Planning', rec))
        print(f"  Resource Planning: {rec}")
        
        self.report_data['recommendations'] = recommendations
        return recommendations

# Initialize Business Intelligence toolkit
if 'all_datasets' in globals() and all_datasets:
    # Collect results from previous analyses if available
    eda_results = eda.insights if 'eda' in globals() else None
    advanced_results = advanced_analysis.insights if 'advanced_analysis' in globals() else None
    
    bi_reporter = HungerHubBusinessIntelligence(
        datasets=all_datasets,
        eda_results=eda_results,
        advanced_results=advanced_results,
        output_dir=notebook_outputs['base']
    )
    
    print("✅ Business Intelligence toolkit initialized")
    print(f"Ready to generate comprehensive reports for {len(all_datasets)} datasets")
else:
    print("❌ Business Intelligence requires processed datasets")
    print("Please run previous sections first")

✅ Business Intelligence toolkit initialized
Ready to generate comprehensive reports for 10 datasets


### 6.1 Executive Summary Report

High-level overview of the data analysis results for executive stakeholders.

In [70]:
# Generate Executive Summary
if 'bi_reporter' in globals():
    executive_summary = bi_reporter.generate_executive_summary()
else:
    print("❌ Business Intelligence toolkit not available")

EXECUTIVE SUMMARY

DATA INVENTORY:
  Total Datasets Analyzed: 10
  Total Records Processed: 6,000
  Total Data Fields: 255
  Largest Dataset: unified_documentheader (1,000 records)

DATA COMPOSITION:
  Numeric Fields: 32 (12.5%)
  Categorical Fields: 197 (77.3%)
  DateTime Fields: 26 (10.2%)

TOP 5 DATASETS BY SIZE:
  1. unified_documentheader: 1,000 records
  2. unified_rw_org: 1,000 records
  3. choice_DOCUMENTHEADER: 500 records
  4. choice_AMX_DONATION_HEADER: 500 records
  5. choice_AMX_DONATION_LINES: 500 records


### 6.2 Data Quality Assessment

Comprehensive evaluation of data health and readiness for operational use.

In [71]:
# Generate Data Quality Assessment
if 'bi_reporter' in globals():
    quality_assessment = bi_reporter.assess_data_quality()
else:
    print("❌ Business Intelligence toolkit not available")



DATA QUALITY ASSESSMENT

Assessing: choice_DOCUMENTHEADER
------------------------------
  Missing Data: 0.0% (Score: 100.0/100)
  Duplicates: 0.0% (Score: 100.0/100)
  Type Consistency: 100.0% (Score: 100.0/100)
  Overall Quality Score: 100.0/100
  Quality Rating: Excellent

Assessing: choice_AMX_DONATION_HEADER
------------------------------
  Missing Data: 0.4% (Score: 99.6/100)
  Duplicates: 0.0% (Score: 100.0/100)
  Type Consistency: 100.0% (Score: 100.0/100)
  Overall Quality Score: 99.9/100
  Quality Rating: Excellent

Assessing: choice_AMX_DONATION_LINES
------------------------------
  Missing Data: 5.3% (Score: 94.7/100)
  Duplicates: 0.0% (Score: 100.0/100)
  Type Consistency: 100.0% (Score: 100.0/100)
  Overall Quality Score: 98.2/100
  Quality Rating: Excellent

Assessing: choice_RW_ORG
------------------------------
  Missing Data: 3.7% (Score: 96.3/100)
  Duplicates: 0.0% (Score: 100.0/100)
  Type Consistency: 100.0% (Score: 100.0/100)
  Overall Quality Score: 98.8/100

### 6.3 Operational Insights & Strategic Recommendations

Actionable insights and recommendations based on comprehensive data analysis.

In [72]:
# Generate Operational Insights and Recommendations
if 'bi_reporter' in globals():
    operational_insights = bi_reporter.generate_operational_insights()
    strategic_recommendations = bi_reporter.generate_recommendations()
else:
    print("❌ Business Intelligence toolkit not available")



OPERATIONAL INSIGHTS

DATA PATTERNS ANALYSIS:
------------------------------
  Data Integration: Successfully created 3 unified datasets for cross-database analysis
  Data Coverage: Multi-source data available: 4 Choice datasets, 3 Agency datasets
  Data Volume: Substantial data volume (6,000 records) suitable for statistical analysis and trend identification
  Geographic Capability: Geographic data available - enables location-based analysis and mapping
  Temporal Capability: Temporal data available - enables trend analysis and seasonal pattern identification
  Quantitative Capability: Quantitative data available - enables volume and efficiency analysis


STRATEGIC RECOMMENDATIONS

DATA QUALITY IMPROVEMENTS:
------------------------------

ANALYTICS CAPABILITIES:
------------------------------
  Advanced Analytics: Data volume supports advanced analytics - consider implementing machine learning models for predictive insights
  Data Integration: Unified datasets created successfully 

### 6.4 Comprehensive Analysis Summary & Next Steps

Final summary of all analytical findings and recommended next steps.

In [73]:
# Generate Final Analysis Summary
if 'bi_reporter' in globals():
    print("COMPREHENSIVE ANALYSIS SUMMARY")
    print("=" * 50)
    
    # Collect all insights from previous analyses
    all_insights = []
    
    # EDA insights
    if 'eda' in globals() and hasattr(eda, 'insights'):
        all_insights.extend([(insight['category'], insight['insight']) for insight in eda.insights])
    
    # Advanced analysis insights
    if 'advanced_analysis' in globals() and hasattr(advanced_analysis, 'insights'):
        all_insights.extend([(insight['category'], insight['insight']) for insight in advanced_analysis.insights])
    
    # BI insights
    all_insights.extend(bi_reporter.report_data['operational_insights'])
    
    print(f"\nTOTAL INSIGHTS GENERATED: {len(all_insights)}")
    print("\nKEY FINDINGS BY CATEGORY:")
    print("-" * 30)
    
    # Group insights by category
    insights_by_category = {}
    for category, insight in all_insights:
        if category not in insights_by_category:
            insights_by_category[category] = []
        insights_by_category[category].append(insight)
    
    # Display key findings
    for category, insights in insights_by_category.items():
        print(f"\n{category.upper()} ({len(insights)} findings):")
        for i, insight in enumerate(insights[:3], 1):  # Show top 3 per category
            print(f"  {i}. {insight}")
        if len(insights) > 3:
            print(f"     ... and {len(insights)-3} more")
    
    # Analysis completion status
    print("\n\nANALYSIS COMPLETION STATUS:")
    print("=" * 40)
    
    analysis_stages = [
        ('Data Extraction', 'Oracle database connection and data extraction'),
        ('ETL Pipeline', 'Data cleaning, transformation, and standardization'),
        ('Univariate Analysis', 'Individual variable analysis and distributions'),
        ('Bivariate Analysis', 'Correlation and relationship analysis'),
        ('Advanced Visualizations', 'Interactive charts and plots'),
        ('Business Intelligence', 'Executive reports and recommendations')
    ]
    
    for stage, description in analysis_stages:
        print(f"  ✅ {stage}: {description}")
    
    # Next steps
    print("\n\nRECOMMENDED NEXT STEPS:")
    print("=" * 30)
    
    next_steps = [
        "Deploy findings in production dashboard environment",
        "Implement automated data pipelines based on ETL framework",
        "Establish regular data quality monitoring",
        "Develop predictive models using correlation insights",
        "Create stakeholder-specific reporting dashboards",
        "Schedule regular analysis updates and reviews"
    ]
    
    for i, step in enumerate(next_steps, 1):
        print(f"  {i}. {step}")
    
    # Save comprehensive report
    final_report = {
        'analysis_date': datetime.now().isoformat(),
        'executive_summary': bi_reporter.report_data['executive_summary'],
        'data_quality': bi_reporter.report_data['data_quality'],
        'operational_insights': bi_reporter.report_data['operational_insights'],
        'recommendations': bi_reporter.report_data['recommendations'],
        'all_insights': all_insights,
        'analysis_stages_completed': [stage for stage, _ in analysis_stages],
        'next_steps': next_steps
    }
    
    report_file = bi_reporter.reports_dir / 'hungerhub_comprehensive_analysis_report.json'
    with open(report_file, 'w') as f:
        json.dump(final_report, f, indent=2, default=str)
    
    print(f"\n✅ ANALYSIS COMPLETE")
    print(f"Comprehensive report saved: {report_file}")
    print(f"All visualizations saved to: {bi_reporter.output_dir / 'visualizations'}")
    print(f"All processed data saved to: {bi_reporter.output_dir / 'csv'} and {bi_reporter.output_dir / 'parquet'}")
    
    print("\nTHANK YOU FOR USING THE HUNGERHUB COMPREHENSIVE ANALYSIS NOTEBOOK!")
    print("This analysis provides a complete Oracle-to-Graphics workflow for data-driven")
    print("decision making in hunger relief operations.")

else:
    print("❌ Cannot generate final summary - Business Intelligence toolkit not available")

COMPREHENSIVE ANALYSIS SUMMARY

TOTAL INSIGHTS GENERATED: 120

KEY FINDINGS BY CATEGORY:
------------------------------

DISTRIBUTION (20 findings):
  1. status in choice_DOCUMENTHEADER is highly right-skewed (skewness: 7.58)
  2. notify in choice_DOCUMENTHEADER is highly right-skewed (skewness: 8.62)
  3. quantity in choice_AMX_DONATION_LINES is highly right-skewed (skewness: 10.01)
     ... and 17 more

DATA QUALITY (20 findings):
  1. documentid in choice_DOCUMENTHEADER has very high uniqueness (100.0%) - possibly an identifier
  2. refnumber in choice_DOCUMENTHEADER has very high uniqueness (100.0%) - possibly an identifier
  3. donationnumber in choice_AMX_DONATION_HEADER has very high uniqueness (100.0%) - possibly an identifier
     ... and 17 more

DATA PATTERN (43 findings):
  1. userid in choice_DOCUMENTHEADER is dominated by 'Unknown' (88.0% of records)
  2. dcontactfax in choice_AMX_DONATION_HEADER is dominated by 'Unknown' (88.2% of records)
  3. sys_flags in choice_RW_ORG

In [74]:
# Analysis execution timer - END
analysis_end_time = datetime.now()
elapsed_time = analysis_end_time - analysis_start_time

# Format elapsed time as H:M:S
hours, remainder = divmod(elapsed_time.total_seconds(), 3600)
minutes, seconds = divmod(remainder, 60)

print("\n" + "=" * 60)
print("ANALYSIS EXECUTION COMPLETED")
print("=" * 60)
print(f"Start time: {analysis_start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"End time: {analysis_end_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{int(seconds):02d}")
print(f"Notebook type: SAMPLE DATA ANALYSIS")
print("\nSample-based analysis completed successfully!")
print("For production analysis with complete data, use: hungerhub_full_data_analysis.ipynb")


ANALYSIS EXECUTION COMPLETED
Start time: 2025-08-09 16:56:17
End time: 2025-08-09 16:56:36
Elapsed time: 00:00:19
Notebook type: SAMPLE DATA ANALYSIS

Sample-based analysis completed successfully!
For production analysis with complete data, use: hungerhub_full_data_analysis.ipynb
